# Sensitivity Matrix, V-Modes & Double-Zernike Analysis

**Author:** Aaron Roodman
**Date Created:** 2026-03-08
**Last Modified:** 2026-06-11
**Status:** In Progress
**Keywords:** Rubin Observatory, AOS, Sensitivity Matrix, SVD, V-modes, Normalization, Double-Zernike, Control

## Description

Comprehensive analysis of the AOS sensitivity matrix: its SVD decomposition,
v-mode properties, normalization schemes, and double-Zernike structure.  This
notebook consolidates the former `smatrix_vmode_info`, `normalization_study`,
and `smatrix_doublez` notebooks into one.

Shared SVD / normalization / double-Zernike primitives live in
[`code/ofc_svd.py`](code/ofc_svd.py) (`compute_normalization_components`,
`normalization_weights_by_scheme`, `focal_zernike_at_points`,
`make_dz_basis_vector`, `decompose_into_dz`, `LABELS_50DOF`, `DOF_UNITS_50`).
The SVD itself is built with OFC's `StateEstimator` (the canonical 4-WFS
construction), validated against a custom SVD.

Sections:
1. Sensitivity Matrix (raw A and normalized Ã)
2. SVD Decomposition (+ StateEstimator validation)
3. V-Mode Analysis (DOF composition)
4. V-Mode Wavefront Signatures
5. Control Equations
6. Noise Amplification & Information Content
7. Control Gain Analysis
8. Comparison Across DOF Sets
9. DOF Set Comparisons (summary)
10. Double-Zernike V-Mode Decomposition (at the 4 WFS)
11. Normalization Scheme Deep-Dive & Unit Invariance *(from normalization_study)*
12. Double-Zernike Field Patterns & Physical Impact *(from smatrix_doublez)*

**Output:** A multi-page PDF plus inline plots/tables characterizing AOS v-modes,
normalization, noise, control behavior, and double-Zernike field structure.

**Conventions:** V-modes numbered from 1; truncation per DOF set (see config).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-03-08 | Aaron Roodman | Initial version (smatrix_vmode_info) |
| 2026-03-14 | Aaron Roodman | Per-Zernike noise from covM86, control-gain delay convergence, physical DOF coeffs |
| 2026-03-25 | Aaron Roodman | Use StateEstimator for SVD/DOF↔v-mode; custom-SVD validation; DOF sets updated |
| 2026-04-07 | Aaron Roodman | v13 OFC config; standard_24; r_j/f_j components; FWHM-only normalization |
| 2026-06-11 | Aaron Roodman | Consolidated normalization_study (§11) and smatrix_doublez (§12); moved shared primitives to code/ofc_svd.py |

## Index Notation Conventions

| Symbol | Meaning | Range |
|--------|---------|-------|
| $i$ | Degree of Freedom (DOF) index | 0–49 (or active subset) |
| $j$ | Pupil-plane Zernike index (Noll) | 4–26 |
| $k$ | Focal-plane (double-Zernike) index | 1–6 |
| $m$ | V-mode index | 1–$N_\text{trunc}$ |

**Key quantities:** $A_{ji}$ sensitivity matrix; $n_i$ normalization weight;
$\tilde{A} = A\,\mathrm{diag}(n_i)$; $V_{im}$ v-mode matrix; $\sigma_m$ singular value;
$a_m$ v-mode amplitude; $(j,k)$ double-Zernike term.

## Table of Contents

1. [Imports](#Imports) · [Configuration](#Configuration) · [Custom Code](#Code)
2. [Sensitivity Matrix](#Sensitivity%20Matrix)
3. [SVD Decomposition](#SVD%20Decomposition)
4. [V-Mode Analysis](#V-Mode%20Analysis)
5. [Wavefront Signatures](#Zernike%20Patterns)
6. [Control Equations](#Control%20Equations)
7. [Noise Amplification](#Noise%20Amplification)
8. [Control Gain](#Control%20Gain)
9. [DOF Set Comparison](#DOF%20Set%20Comparison)
10. [Double-Zernike (4 WFS)](#Double-Zernike)
11. [Normalization Deep-Dive](#Normalization-Deep-Dive)
12. [Double-Zernike Field Patterns](#DZ-Field-Patterns)

<a id='Imports' ></a>
## Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from matplotlib import colors, cm
from matplotlib.backends.backend_pdf import PdfPages
import scipy.io

from lsst.ts.ofc import (OFC, OFCData, SensitivityMatrix, StateEstimator,
                         BendModeToForce)
from lsst.ts.wep.utils import convertZernikesToPsfWidth

# Shared SVD / normalization / double-Zernike primitives.
sys.path.insert(0, str(Path.cwd() / 'code'))
import ofc_svd as osv

%load_ext autoreload
%autoreload 2
%matplotlib inline

<a id='Configuration' ></a>
## Configuration

In [ ]:
# --- OFC config path ---
ofc_config_dir = '/home/r/roodman/u/LSST/packages/ts_config_mttcs/MTAOS/v13/ofc'

# --- DOF labels (all 50) ---
labels_50dof = [
    'M2_dz', 'M2_dx', 'M2_dy', 'M2_rx', 'M2_ry',
    'Cam_dz', 'Cam_dx', 'Cam_dy', 'Cam_rx', 'Cam_ry',
    'B1_1', 'B1_2', 'B1_3', 'B1_4', 'B1_5',
    'B1_6', 'B1_7', 'B1_8', 'B1_9', 'B1_10',
    'B1_11', 'B1_12', 'B1_13', 'B1_14', 'B1_15',
    'B1_16', 'B1_17', 'B1_18', 'B1_19', 'B1_20',
    'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5',
    'B2_6', 'B2_7', 'B2_8', 'B2_9', 'B2_10',
    'B2_11', 'B2_12', 'B2_13', 'B2_14', 'B2_15',
    'B2_16', 'B2_17', 'B2_18', 'B2_19', 'B2_20',
]

# --- DOF subsets (must be sorted — OFCData.dof_idx returns sorted indices) ---
# hexapod_10:   Camera and M2 hexapods only (5+5 rigid body DOFs)
# standard_21:  Hexapods (without M2_dz) + 7 M1M3 bending modes + 5 M2 bending modes
# standard_22:  AOS operational default for months — adds M2_dz to standard_21
# standard_24:  Adds M2 bending modes B2_6 and B2_7 to standard_22
# extended_25:  Adds 3 higher-order mirror modes (B1_12, B1_20, B2_18) — may be
#               difficult to control with only the 4 corner WFS
# all_50:       All 50 degrees of freedom
dof_sets = {
    'hexapod_10': list(range(0, 10)),
    'standard_21': sorted(list(range(1, 17)) + list(range(30, 35))),
    'standard_22': sorted(list(range(0, 17)) + list(range(30, 35))),
    'standard_24': sorted(list(range(0, 17)) + list(range(30, 37))),
    'extended_25': sorted(list(range(0, 17)) + list(range(30, 35)) + [21, 29, 47]),
    'all_50': list(range(0, 50)),
}

# --- Zernike selection ---
zn = np.array([4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 26])
zn_idx = zn - 4
n_zernike = len(zn_idx)

# --- WFS corners ---
sensor_name_list = ['R00_SW0', 'R04_SW0', 'R40_SW0', 'R44_SW0']
n_wfs = len(sensor_name_list)

# --- Default DOF set for detailed analysis ---
default_dof_set = 'standard_24'

# --- V-mode truncation: how many modes to retain for each DOF set ---
n_modes_truncated = {
    'hexapod_10': 10,
    'standard_21': 12,
    'standard_22': 12,
    'standard_24': 14,
    'extended_25': 20,
    'all_50': 20,
}

# --- Normalization weight mode ---
# All schemes set n_j so that Atilde = A @ diag(n_j).  Only some are invariant
# under x_j -> alpha * x_j (a change of physical units for DOF j); see the
# Normalization Weights section.
#
# Options:
#   'default'   — stored normalization weights as-is        (NOT unit-invariant)
#   'rf'        — computed r_j * f_j                        (NOT unit-invariant)
#   'r_only'    — range weights only:        n_j = r_j      (unit-invariant)
#   'inv_f'     — inverse FWHM weights:      n_j = 1/f_j    (unit-invariant)
#   'geom_mean' — geometric mean: n_j = sqrt(r_j / f_j)      (unit-invariant; recommended)
#   'tunable'   — n_j = r_j**a * f_j**(a-1), a = norm_a      (unit-invariant any a)
norm_mode = 'default'
# Exponent for 'tunable' mode: a=0 -> 1/f, a=0.5 -> sqrt(r/f), a=1 -> r_only
norm_a = 0.5

# --- Control loop options ---
# skip_mode: for N-step delay, only apply corrections every N images
# (measure on image 1, apply before image 1+N, skip images 2..N)
use_skip_mode = True

# --- Markers for multi-line plots ---
marker_cycle = ['o', 's', '^', 'D', 'v', 'P', '*', 'X', 'h', 'p', '<', '>']

print(f'Zernike terms: {zn} ({n_zernike} terms)')
print(f'WFS sensors: {sensor_name_list} ({n_wfs} sensors)')
print(f'Measurement vector size: {n_zernike} x {n_wfs} = {n_zernike * n_wfs}')
for name, dofs in dof_sets.items():
    ntrunc = n_modes_truncated[name]
    print(f'DOF set "{name}": {len(dofs)} DOFs, truncated to {ntrunc} v-modes')

<a id='Code' ></a>
## Custom Code

Helper functions that supplement `StateEstimator` functionality.
These are used by the analysis sections below.

In [ ]:
def make_comp_dof_idx(dof_idx_list):
    """Convert a list of active DOF indices (0-49) to the comp_dof_idx dict
    expected by OFCData.comp_dof_idx setter.

    Component layout (50 DOFs total):
        m2HexPos:  indices 0–4   (5)
        camHexPos: indices 5–9   (5)
        M1M3Bend:  indices 10–29 (20)
        M2Bend:    indices 30–49 (20)
    """
    dof_set = set(dof_idx_list)
    components = {
        'm2HexPos':  (0, 5),
        'camHexPos': (5, 5),
        'M1M3Bend':  (10, 20),
        'M2Bend':    (30, 20),
    }
    return {
        name: np.array([start + i in dof_set for i in range(length)], dtype=bool)
        for name, (start, length) in components.items()
    }


def vmode_amplitudes(z_measured, svd_result):
    """Compute v-mode amplitudes: a_m = U^T_k @ z_measured.

    Parameters
    ----------
    z_measured : array (n_z * n_wfs,)
        Concatenated Zernike measurements from all WFS.
    svd_result : dict
        SVD result dict with keys U, s, V.

    Returns
    -------
    a : array (n_modes,)
        V-mode amplitudes in microns of wavefront.
    """
    return svd_result['U'].T @ z_measured


def dimensionless_dof_correction(z_measured, svd_result, n_modes_use=None):
    """Compute dimensionless DOF correction: q = V @ Sigma^-1 @ U^T @ z.

    Parameters
    ----------
    z_measured : array (n_z * n_wfs,)
    svd_result : dict
    n_modes_use : int or None
        Number of modes to retain (truncation). None = all.

    Returns
    -------
    q : array (n_dof,)
        Dimensionless DOF corrections.
    """
    a = vmode_amplitudes(z_measured, svd_result)
    s = svd_result['s']
    V = svd_result['V']
    if n_modes_use is None:
        n_modes_use = len(s)
    return V[:, :n_modes_use] @ (a[:n_modes_use] / s[:n_modes_use])


def physical_dof_correction(z_measured, svd_result, norm_vec, n_modes_use=None):
    """Compute physical DOF correction: delta_dof_j = n_i * q_j.

    Parameters
    ----------
    z_measured : array (n_z * n_wfs,)
    svd_result : dict
    norm_vec : array (n_dof_full,)
        Full normalization vector (50 entries).
    n_modes_use : int or None

    Returns
    -------
    delta_dof : array (n_dof,)
        Physical DOF corrections in microns or degrees.
    """
    q = dimensionless_dof_correction(z_measured, svd_result, n_modes_use)
    n_sub = norm_vec[svd_result['dof_indices']]
    return n_sub * q


print('Custom code functions defined.')

# --- Stdout tee for PDF capture ---
import sys

_tee_buf = None
_tee_old_stdout = None

class _Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data)
    def flush(self):
        for s in self.streams:
            s.flush()

def pdf_tee_start():
    """Start capturing stdout to buffer while still printing to console."""
    global _tee_buf, _tee_old_stdout
    _tee_buf = io.StringIO()
    _tee_old_stdout = sys.stdout
    sys.stdout = _Tee(_tee_old_stdout, _tee_buf)

def pdf_tee_stop():
    """Stop capturing stdout and write captured text to PDF."""
    global _tee_buf, _tee_old_stdout
    sys.stdout = _tee_old_stdout
    _text = _tee_buf.getvalue()
    if _text.strip():
        pdf_text_page(_text, fontsize=9)
    _tee_buf = None
    _tee_old_stdout = None

# Range/FWHM normalization-weight components now come from ofc_svd.
from ofc_svd import compute_normalization_components

In [ ]:
# --- PDF output setup ---
import io, contextlib

_wgt_tag_map = {'default': 'defwgt', 'rf': 'rfwgt', 'r_only': 'rwgt',
                'inv_f': 'invfwgt', 'geom_mean': 'gmwgt',
                'tunable': f'awgt{norm_a:.2f}'}
wgt_tag = _wgt_tag_map[norm_mode]
pdf_filename = f'output/smatrix_vmode_info_{default_dof_set}_{wgt_tag}.pdf'

pdf = PdfPages(pdf_filename)
print(f'PDF output: {pdf_filename}')

def pdf_text_page(text, fontsize=14):
    """Add a text-only page to the PDF."""
    fig_pdf = plt.figure(figsize=(11, 8.5))
    fig_pdf.text(0.05, 0.95, text, transform=fig_pdf.transFigure,
                 fontsize=fontsize, verticalalignment='top',
                 fontfamily='monospace', wrap=True)
    plt.axis('off')
    pdf.savefig(fig_pdf)
    plt.close(fig_pdf)

def pdf_section(title, body=''):
    """Add a section header page to the PDF."""
    text = title + '\n' + '=' * len(title)
    if body:
        text += '\n\n' + body
    pdf_text_page(text, fontsize=16)

def pdf_save(fig):
    """Save current figure to PDF."""
    pdf.savefig(fig, bbox_inches='tight')

def pdf_capture_print(func):
    """Run func, capture stdout, print it, and add as PDF text page."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        func()
    text = buf.getvalue()
    print(text)
    pdf_text_page(text, fontsize=9)

<a id='Sensitivity Matrix' ></a>
## 1. Sensitivity Matrix

Build the raw sensitivity matrix **A** and the normalized matrix **Ã = A @ diag(n_j)**.

- **A** maps physical DOF motions to Zernike wavefront changes (units: microns/Zernike per physical DOF unit)
- **Ã** normalizes each DOF column by n_i = r_j × f_j, placing all DOFs on equal (dimensionless) footing

In [ ]:
pdf_section('1. Sensitivity Matrix')

# --- Compute normalization components from a temporary OFCData ---
_ofc_tmp = OFCData('lsst', config_dir=ofc_config_dir)
_ofc_tmp.zn_selected = zn
field_angles = [_ofc_tmp.sample_points[sensor] for sensor in sensor_name_list]
_dz_sens_tmp = SensitivityMatrix(_ofc_tmp)
range_weights, fwhm_weights = compute_normalization_components(
    _ofc_tmp, _dz_sens_tmp, field_angles)
norm_default = _ofc_tmp.normalization_weights.copy()
del _ofc_tmp, _dz_sens_tmp

# --- Build the normalization vector according to norm_mode ---
if norm_mode == 'default':
    norm_50 = norm_default
    print(f'Normalization mode: default (stored weights)')
elif norm_mode == 'rf':
    norm_50 = range_weights * fwhm_weights
    print(f'Normalization mode: rf (computed r_j * f_j)')
elif norm_mode == 'r_only':
    norm_50 = range_weights
    print(f'Normalization mode: r_only (range weights only)')
elif norm_mode == 'inv_f':
    norm_50 = 1.0 / fwhm_weights
    print(f'Normalization mode: inv_f (1 / f_j)')
elif norm_mode == 'geom_mean':
    norm_50 = np.sqrt(range_weights / fwhm_weights)
    print(f'Normalization mode: geom_mean (sqrt(r_j / f_j), unit-invariant)')
elif norm_mode == 'tunable':
    norm_50 = (range_weights ** norm_a) * (fwhm_weights ** (norm_a - 1.0))
    print(f'Normalization mode: tunable a={norm_a} '
          f'(n_j = r_j**{norm_a} * f_j**{norm_a - 1.0}, unit-invariant)')
else:
    raise ValueError(f'Unknown norm_mode: {norm_mode!r}')

# --- Create the OFCData object used everywhere ---
ofc_data = OFCData('lsst', config_dir=ofc_config_dir)
ofc_data.zn_selected = zn
ofc_data.normalization_weights = norm_50

print('Field angles (deg):', field_angles)

dz_sensitivity_matrix = SensitivityMatrix(ofc_data)
sens_3d = dz_sensitivity_matrix.evaluate(field_angles, 0.0)
print(f'Raw 3D sensitivity matrix shape: {sens_3d.shape}  (n_wfs, n_zernike_full, n_dof_full)')

# Select Zernike subset and reshape to 2D: rows = (Zernike x WFS), cols = DOFs
sens_3d = sens_3d[:, zn_idx, :]
A_full = sens_3d.reshape((-1, sens_3d.shape[2]))

# Select active DOF indices
A_full = A_full[:, ofc_data.dof_idx]
print(f'Raw sensitivity matrix A shape: {A_full.shape}  (n_zernike*n_wfs, n_dof)')

# Normalization vector n_j for each DOF
norm_vector = ofc_data.normalization_weights[ofc_data.dof_idx]
print(f'Normalization vector shape: {norm_vector.shape}')

# Normalized sensitivity matrix: Atilde = A @ diag(n_j)
Atilde_full = A_full @ np.diag(norm_vector)
print(f'Normalized sensitivity matrix Atilde shape: {Atilde_full.shape}')


### 1.1 Sensitivity Matrix Visualization

In [ ]:
pdf_section('1.1 Sensitivity Matrix Visualization')
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw A (first WFS only for clarity)
r0 = axes[0].imshow(A_full[:n_zernike, :], aspect='auto')
axes[0].set_title('Raw A (WFS 0)')
axes[0].set_ylabel('Zernike index')
axes[0].set_yticks(range(0, n_zernike, 5), [f'z{zn[i]}' for i in range(0, n_zernike, 5)])
xt = np.arange(0, 50, 10)
axes[0].set_xticks(xt, [labels_50dof[i] for i in xt], rotation=45, ha='right')
fig.colorbar(r0, ax=axes[0], shrink=0.8)

# Normalized Atilde (first WFS only)
r1 = axes[1].imshow(Atilde_full[:n_zernike, :], aspect='auto')
axes[1].set_title('Normalized Ã (WFS 0)')
axes[1].set_ylabel('Zernike index')
axes[1].set_yticks(range(0, n_zernike, 5), [f'z{zn[i]}' for i in range(0, n_zernike, 5)])
axes[1].set_xticks(xt, [labels_50dof[i] for i in xt], rotation=45, ha='right')
fig.colorbar(r1, ax=axes[1], shrink=0.8)

fig.tight_layout()
pdf_save(fig)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# |A| log scale
r0 = axes[0].imshow(np.abs(A_full[:n_zernike, :]), aspect='auto', norm=colors.LogNorm())
axes[0].set_title('|A| log scale (WFS 0)')
axes[0].set_ylabel('Zernike index')
axes[0].set_yticks(range(0, n_zernike, 5), [f'z{zn[i]}' for i in range(0, n_zernike, 5)])
xt = np.arange(0, 50, 10)
axes[0].set_xticks(xt, [labels_50dof[i] for i in xt], rotation=45, ha='right')
fig.colorbar(r0, ax=axes[0], shrink=0.8)

# |Atilde| log scale
r1 = axes[1].imshow(np.abs(Atilde_full[:n_zernike, :]), aspect='auto', norm=colors.LogNorm())
axes[1].set_title('|Ã| log scale (WFS 0)')
axes[1].set_ylabel('Zernike index')
axes[1].set_yticks(range(0, n_zernike, 5), [f'z{zn[i]}' for i in range(0, n_zernike, 5)])
axes[1].set_xticks(xt, [labels_50dof[i] for i in xt], rotation=45, ha='right')
fig.colorbar(r1, ax=axes[1], shrink=0.8)

fig.tight_layout()
pdf_save(fig)
plt.show()

### 1.2 Normalization Weights

In [ ]:
pdf_section(f'1.2 Normalization Weights (mode: {norm_mode})')
fig, ax = plt.subplots(1, 1, figsize=(14, 4))
ax.semilogy(norm_vector, 'o-', markersize=4)
ax.set_xlabel('DOF index')
ax.set_ylabel('Normalization weight n_j')
ax.set_title(f'DOF Normalization Weights (mode: {norm_mode})')
# Mark boundaries
for xv in [5, 10, 30]:
    ax.axvline(xv - 0.5, color='gray', ls='--', alpha=0.5)
ax.set_xticks([2.5, 7.5, 20, 40], ['M2 hex', 'Cam hex', 'M1M3 bending', 'M2 bending'])
ax.grid(alpha=0.3)
fig.tight_layout()
pdf_save(fig)
plt.show()

### 1.3 Normalization Weight Comparison

The normalization weight for each DOF is n_i = r_j × f_j, where:
- **r_j (range weight):** physical range of each DOF — hexapod stroke or bending mode force range divided by max rotation matrix element
- **f_j (FWHM weight):** sensitivity of FWHM to each DOF — RSS of the FWHM sensitivity across field positions

These are computed by `generate_normalization_weights` in `ts_ofc`.

In [ ]:
pdf_tee_start()

pdf_section('1.3 Normalization Weight Comparison')

# range_weights and fwhm_weights already computed in build-smatrix
norm_computed = range_weights * fwhm_weights

# Validate computed vs default stored weights
norm_stored = norm_default
max_diff = np.max(np.abs(norm_computed - norm_stored))
print(f'Normalization weight validation: max |computed - stored| = {max_diff:.2e}')
if max_diff < 1e-10:
    print('PASS: computed r_j*f_j matches default stored weights')
else:
    print('WARNING: mismatch between computed r_j*f_j and default stored weights')
    for i in range(50):
        if abs(norm_computed[i] - norm_stored[i]) > 1e-10:
            ratio = norm_computed[i] / norm_stored[i] if norm_stored[i] != 0 else float('inf')
            print(f'  DOF {i:2d} ({labels_50dof[i]:>20s}): '
                  f'computed={norm_computed[i]:.6e}, stored={norm_stored[i]:.6e}, '
                  f'ratio={ratio:.4f}')

print(f'\nActive normalization mode: {norm_mode} (wgt_tag={wgt_tag})')

pdf_tee_stop()


In [ ]:
# Plot range weights, FWHM weights, and combined
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].semilogy(range_weights, 'o-', markersize=4, color='steelblue')
axes[0].set_ylabel('Range weight r_j')
axes[0].set_title('Range Weights r_j (physical stroke / max force-to-mode)')
for xv in [5, 10, 30]:
    axes[0].axvline(xv - 0.5, color='gray', ls='--', alpha=0.5)
axes[0].grid(alpha=0.3)

axes[1].semilogy(fwhm_weights, 'o-', markersize=4, color='coral')
axes[1].set_ylabel('FWHM weight f_j')
axes[1].set_title('FWHM Weights f_j (RSS of FWHM sensitivity across field)')
for xv in [5, 10, 30]:
    axes[1].axvline(xv - 0.5, color='gray', ls='--', alpha=0.5)
axes[1].grid(alpha=0.3)

axes[2].semilogy(norm_vector, 'o-', markersize=4, color='green')
axes[2].set_ylabel('n_i = r_j * f_j')
axes[2].set_title('Combined Normalization Weights n_i = r_j * f_j')
for xv in [5, 10, 30]:
    axes[2].axvline(xv - 0.5, color='gray', ls='--', alpha=0.5)
axes[2].grid(alpha=0.3)

axes[2].set_xlabel('DOF index')
axes[2].set_xticks([2.5, 7.5, 20, 40], ['M2 hex', 'Cam hex', 'M1M3 bending', 'M2 bending'])

fig.tight_layout()
pdf_save(fig)
plt.show()

# Normalized sensitivity matrix comparison: stored vs computed weights (WFS 0)
dof_idx = ofc_data.dof_idx
n_computed = (range_weights * fwhm_weights)[dof_idx]
n_stored = norm_default[dof_idx]
Atilde_stored = A_full @ np.diag(n_stored)
Atilde_computed = A_full @ np.diag(n_computed)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Stored weights
vmax = max(np.abs(Atilde_stored[:n_zernike, :]).max(),
           np.abs(Atilde_computed[:n_zernike, :]).max())
im0 = axes[0].imshow(Atilde_stored[:n_zernike, :], aspect='auto',
                      vmin=-vmax, vmax=vmax, cmap='RdBu_r')
axes[0].set_title('Ã default weights (WFS 0)')
axes[0].set_ylabel('Zernike index')
axes[0].set_yticks(range(0, n_zernike, 5), [f'z{zn[i]}' for i in range(0, n_zernike, 5)])
fig.colorbar(im0, ax=axes[0], shrink=0.8)

# Computed weights
im1 = axes[1].imshow(Atilde_computed[:n_zernike, :], aspect='auto',
                      vmin=-vmax, vmax=vmax, cmap='RdBu_r')
axes[1].set_title('Ã computed weights (WFS 0)')
axes[1].set_ylabel('Zernike index')
axes[1].set_yticks(range(0, n_zernike, 5), [f'z{zn[i]}' for i in range(0, n_zernike, 5)])
fig.colorbar(im1, ax=axes[1], shrink=0.8)

# Difference
diff_matrix = Atilde_computed[:n_zernike, :] - Atilde_stored[:n_zernike, :]
dmax = np.abs(diff_matrix).max()
if dmax < 1e-15:
    dmax = 1.0  # avoid zero range
im2 = axes[2].imshow(diff_matrix, aspect='auto',
                      vmin=-dmax, vmax=dmax, cmap='RdBu_r')
axes[2].set_title(f'Difference (max={dmax:.2e})')
axes[2].set_ylabel('Zernike index')
axes[2].set_yticks(range(0, n_zernike, 5), [f'z{zn[i]}' for i in range(0, n_zernike, 5)])
fig.colorbar(im2, ax=axes[2], shrink=0.8)

# DOF labels on x-axis
n_dof = Atilde_stored.shape[1]
xt = np.linspace(0, n_dof - 1, min(n_dof, 10), dtype=int)
dof_labels = [labels_50dof[dof_idx[j]] for j in xt]
for ax in axes:
    ax.set_xticks(xt)
    ax.set_xticklabels(dof_labels, rotation=45, ha='right', fontsize=8)

fig.suptitle('Normalized Sensitivity Matrix: Stored vs Computed Weights', fontsize=13)
fig.tight_layout()
pdf_save(fig)
plt.show()


### 1.4 Inspect Individual Sensitivity Matrix Elements

Look up a specific (DOF, Zernike, WFS) element of the sensitivity matrix,
showing the raw value and normalized values using both stored and computed weights.

In [ ]:
# --- Configuration: choose DOF, Zernike, and WFS to inspect ---
inspect_dof_name = 'Cam_rx'      # DOF name from labels_50dof
inspect_zernike = 5              # Noll index
inspect_wfs = 'R00_SW0'          # WFS sensor name

pdf_tee_start()
pdf_section('1.4 Inspect Individual Sensitivity Matrix Elements')

# Resolve indices
dof_idx_full = labels_50dof.index(inspect_dof_name)  # 0-49
dof_idx_active = list(ofc_data.dof_idx).index(dof_idx_full)  # column in A_full
zk_pos = list(zn).index(inspect_zernike)  # row within one WFS block
wfs_idx = sensor_name_list.index(inspect_wfs)  # WFS block index

row = wfs_idx * n_zernike + zk_pos

# Raw sensitivity matrix element
a_raw = A_full[row, dof_idx_active]

# Normalization weights
n_stored = norm_default[dof_idx_full]
n_computed = (range_weights * fwhm_weights)[dof_idx_full]
r_j = range_weights[dof_idx_full]
f_j = fwhm_weights[dof_idx_full]

# Normalized values
a_norm_stored = a_raw * n_stored
a_norm_computed = a_raw * n_computed

print(f'DOF:     {inspect_dof_name} (index {dof_idx_full})')
print(f'Zernike: z{inspect_zernike} (Noll)')
print(f'WFS:     {inspect_wfs} (index {wfs_idx})')
print()
print(f'Raw sensitivity A[z{inspect_zernike}, {inspect_dof_name}] = {a_raw:.6e} µm/unit')
print()
print(f'Normalization weights for {inspect_dof_name}:')
print(f'  Range weight   r_j = {r_j:.6e}')
print(f'  FWHM weight    f_j = {f_j:.6e}')
print(f'  Stored   n_j       = {n_stored:.6e}')
print(f'  Computed r_j * f_j = {n_computed:.6e}')
if n_stored != 0:
    print(f'  Ratio computed/stored = {n_computed / n_stored:.6f}')
print()
print(f'Normalized Ã[z{inspect_zernike}, {inspect_dof_name}]:')
print(f'  With stored weights:   {a_norm_stored:.6e}')
print(f'  With computed weights: {a_norm_computed:.6e}')
if a_norm_stored != 0:
    print(f'  Ratio: {a_norm_computed / a_norm_stored:.6f}')

# Also show all WFS values for this DOF and Zernike
print(f'\nAll WFS values for z{inspect_zernike} x {inspect_dof_name}:')
print(f'  {"WFS":>10s}  {"A (raw)":>12s}  {"Ã stored":>12s}  {"Ã computed":>12s}')
for iwfs, wfs_name in enumerate(sensor_name_list):
    r = iwfs * n_zernike + zk_pos
    a_val = A_full[r, dof_idx_active]
    print(f'  {wfs_name:>10s}  {a_val:>12.6e}  {a_val * n_stored:>12.6e}  {a_val * n_computed:>12.6e}')

# Show all Zernike values for this DOF and WFS
print(f'\nAll Zernike values for {inspect_wfs} x {inspect_dof_name}:')
print(f'  {"Zernike":>10s}  {"A (raw)":>12s}  {"Ã stored":>12s}  {"Ã computed":>12s}')
for izk, zn_val in enumerate(zn):
    r = wfs_idx * n_zernike + izk
    a_val = A_full[r, dof_idx_active]
    print(f'  {"z" + str(zn_val):>10s}  {a_val:>12.6e}  {a_val * n_stored:>12.6e}  {a_val * n_computed:>12.6e}')

pdf_tee_stop()


<a id='SVD Decomposition' ></a>
## 2. SVD Decomposition

Decompose the normalized sensitivity matrix: **Ã = U Σ Vᵀ**

- **U** (n_z×n_wfs, n_modes): left singular vectors — Zernike measurement patterns
- **Σ** (n_modes, n_modes): singular values — wavefront change per unit dimensionless DOF motion
- **V** (n_dof, n_modes): right singular vectors — DOF composition of each v-mode

In [ ]:
pdf_tee_start()

pdf_section('2. SVD Decomposition')
# Create StateEstimator for each DOF set using OFC code
# StateEstimator computes SVD internally: U, S, Vh = svd(A[:, dof_idx] @ N)
# Note: StateEstimator's U is in DZ coefficient space (not WFS×Zernike).
# We also compute U_wfs for per-WFS visualization and noise calculations.
state_estimators = {}
svd_results = {}

for name, dof_idx_list in dof_sets.items():
    ofc_i = OFCData('lsst', config_dir=ofc_config_dir)
    ofc_i.comp_dof_idx = make_comp_dof_idx(dof_idx_list)
    ofc_i.zn_selected = zn
    ofc_i.normalization_weights = norm_50
    ofc_i.controller['truncation_index'] = n_modes_truncated[name]

    # Verify DOF mask was set correctly
    assert list(ofc_i.dof_idx) == dof_idx_list, \
        f'{name}: dof_idx mismatch: {list(ofc_i.dof_idx)} != {dof_idx_list}'

    se = StateEstimator(ofc_i)
    state_estimators[name] = se

    labels_sub = [labels_50dof[i] for i in dof_idx_list]
    n_trunc_i = n_modes_truncated[name]
    n_sub = norm_vector[dof_idx_list]

    # U_wfs: project DZ-space SVD onto 4-WFS evaluated sensitivity matrix
    # U_wfs = Atilde_wfs @ V @ diag(1/s), where Atilde_wfs = A_full[:, dof_idx] @ diag(n)
    Atilde_wfs = A_full[:, dof_idx_list] @ np.diag(n_sub)
    U_wfs = Atilde_wfs @ se.Vh.T @ np.diag(1.0 / se.S)

    svd_results[name] = dict(
        U=se.U, s=se.S, V=se.Vh.T, Vh=se.Vh,
        U_wfs=U_wfs,
        dof_indices=dof_idx_list, labels=labels_sub,
        Atilde_sub=se.U @ np.diag(se.S) @ se.Vh,
        Atilde_wfs=Atilde_wfs,
        n_modes=len(se.S),
        n_trunc=n_trunc_i,
        norm_sub=n_sub,
        state_estimator=se,
    )
    r = svd_results[name]
    print(f'{name:15s}: {r["n_modes"]:3d} modes, '
          f'σ_max={r["s"][0]:.2f}, σ_min={r["s"][-1]:.2e}, '
          f'cond={r["s"][0]/r["s"][-1]:.1f}')

se_default = state_estimators[default_dof_set]
r_default = svd_results[default_dof_set]
print(f'\nDefault ({default_dof_set}):')
print(f'  U (DZ space): {se_default.U.shape}')
print(f'  U_wfs (4 WFS): {r_default["U_wfs"].shape}')
print(f'  S: {se_default.S.shape}, Vh: {se_default.Vh.shape}')
print(f'  truncation_index: {se_default.truncate_index}')

# For cells that need WFS-space Zernike dimensions
n_zk_svd = n_zernike
zn_svd = zn

pdf_tee_stop()

### 2.0 Validation: StateEstimator vs Custom SVD

Reproduce the SVD for `standard_22` using the notebook's sensitivity matrix and
compare against StateEstimator's results. Sign flips per mode are expected
(the SVD is unique only up to sign of each singular vector pair).

In [ ]:
pdf_tee_start()

pdf_section('2.0 Validation: StateEstimator vs Custom SVD')
# --- Validation: custom SVD vs StateEstimator for standard_22 ---
dof_idx_22 = dof_sets['standard_22']
n_trunc_22 = n_modes_truncated['standard_22']
r_22 = svd_results['standard_22']
se_22 = r_22['state_estimator']
n_sub_22 = norm_vector[dof_idx_22]

# Build Atilde from the notebook's A_full (evaluated at 4 WFS) for comparison
A_sub = A_full[:, dof_idx_22]
Atilde_notebook = A_sub @ np.diag(n_sub_22)
U_nb, s_nb, Vh_nb = np.linalg.svd(Atilde_notebook, full_matrices=False)
V_nb = Vh_nb.T

# --- 1. Compare sensitivity matrices ---
print('1. Sensitivity matrix shapes:')
print(f'   From StateEstimator (DZ coefficients): {r_22["Atilde_sub"].shape}')
print(f'   From SensitivityMatrix.evaluate (4 WFS): {Atilde_notebook.shape}')
if r_22['Atilde_sub'].shape == Atilde_notebook.shape:
    diff = np.max(np.abs(r_22['Atilde_sub'] - Atilde_notebook))
    print(f'   Max |difference|: {diff:.2e}')
else:
    print('   Different shapes — StateEstimator uses DZ coefficients,')
    print('   notebook uses SensitivityMatrix.evaluate at 4 WFS field angles.')

# --- 2. Compare singular values ---
n_compare = min(n_trunc_22, len(r_22['s']), len(s_nb))
print(f'\n2. Singular value comparison (first {n_compare} modes):')
print(f'{"mode":>5s} {"σ (SE)":>14s} {"σ (4 WFS)":>14s} {"ratio":>10s}')
for m in range(n_compare):
    ratio = r_22['s'][m] / s_nb[m] if s_nb[m] > 0 else np.nan
    print(f'{m+1:>5d} {r_22["s"][m]:>14.4f} {s_nb[m]:>14.4f} {ratio:>10.6f}')

# --- 3. Validate against StateEstimator.get_vmodes_from_dofs ---
np.random.seed(42)
dof_test = np.zeros(50)
dof_test[dof_idx_22] = np.random.randn(len(dof_idx_22))

vmodes_se = se_22.get_vmodes_from_dofs(dof_test)
V_22 = r_22['V']
vmodes_custom = (dof_test[dof_idx_22] / n_sub_22) @ V_22[:, :n_trunc_22]

print(f'\n3. V-mode reconstruction from random DOF state:')
print(f'{"mode":>5s} {"SE":>12s} {"custom":>12s} {"diff":>12s}')
for m in range(n_trunc_22):
    print(f'{m+1:>5d} {vmodes_se[m]:>12.6f} {vmodes_custom[m]:>12.6f} '
          f'{vmodes_se[m] - vmodes_custom[m]:>12.2e}')
max_diff = np.max(np.abs(vmodes_se - vmodes_custom))
print(f'\nMax |vmode_SE - vmode_custom|: {max_diff:.2e}')
if max_diff < 1e-10:
    print('PASS: Custom SVD exactly matches StateEstimator')
elif max_diff < 1e-6:
    print('PASS: Custom SVD matches StateEstimator to numerical precision')
else:
    print('WARNING: Significant difference — check sign flips or matrix source')
    for m in range(n_trunc_22):
        if np.abs(vmodes_se[m] + vmodes_custom[m]) < np.abs(vmodes_se[m] - vmodes_custom[m]):
            print(f'  Mode {m+1}: likely sign flip')

pdf_tee_stop()

### 2.1 Singular Value Spectrum

In [ ]:
pdf_section('2.1 Singular Value Spectrum')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overlay all DOF sets (1-indexed v-modes)
for name, r in svd_results.items():
    axes[0].semilogy(np.arange(1, r['n_modes'] + 1), r['s'], '.-', markersize=5,
                     label=f'{name} ({r["n_modes"]} DOFs)')
axes[0].set_xlabel('V-mode m')
axes[0].set_ylabel('Singular value σ_m')
axes[0].set_title('Singular Value Spectrum')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Right: normalized σ_m / σ_max for the default set
r = svd_results[default_dof_set]
s_norm = r['s'] / r['s'][0]
axes[1].semilogy(np.arange(1, r['n_modes'] + 1), s_norm, 'o-', markersize=5)
for thresh in [1e-2, 1e-3, 1e-4]:
    axes[1].axhline(thresh, color='red', ls='--', alpha=0.4)
    axes[1].text(r['n_modes'], thresh * 1.3, f'{thresh:.0e}', color='red', fontsize=8, ha='right')
n_trunc = n_modes_truncated[default_dof_set]
axes[1].axvline(n_trunc + 0.5, color='green', ls='-', alpha=0.6, label=f'Truncation at {n_trunc}')
axes[1].set_xlabel('V-mode m')
axes[1].set_ylabel('σ_m / σ_max')
axes[1].set_title(f'Normalized Spectrum ({default_dof_set})')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

fig.tight_layout()
pdf_save(fig)
plt.show()

### 2.2 Mode Truncation

Modes with σ_m < threshold × σ_max are truncated. Show how many modes survive at each threshold.

In [ ]:
pdf_tee_start()

pdf_section('2.2 Mode Truncation')
thresholds = [1e-2, 1e-3, 1e-4, 1e-5]

print(f'{"DOF set":>15s} {"n_DOF":>6s} {"n_trunc":>8s}', end='')
for t in thresholds:
    print(f'  thresh={t:.0e}', end='')
print()

for name, r in svd_results.items():
    n_trunc = n_modes_truncated[name]
    print(f'{name:>15s} {len(r["dof_indices"]):>6d} {n_trunc:>8d}', end='')
    for t in thresholds:
        n_kept = np.sum(r['s'] > t * r['s'][0])
        print(f'  {n_kept:>12d}', end='')
    print()

print(f'\nRetained v-modes per DOF set:')
for name, n_trunc in n_modes_truncated.items():
    r = svd_results[name]
    print(f'  {name}: v-modes 1–{n_trunc} of {r["n_modes"]}  '
          f'(σ_{n_trunc}/σ_1 = {r["s"][n_trunc-1]/r["s"][0]:.2e})')

pdf_tee_stop()

<a id='V-Mode Analysis' ></a>
## 3. V-Mode Analysis

Each column of V defines a v-mode: a unit vector in DOF space.
V_im is the contribution of dimensionless DOF j to v-mode m.

### 3.1 V-Mode Composition in DOF Space

In [ ]:
pdf_section('3. V-Mode Analysis')
r = svd_results[default_dof_set]
V = r['V']
s = r['s']
n_dof_sub = len(r['dof_indices'])
labels_sub = r['labels']
n_trunc = r['n_trunc']

fig = plt.figure(figsize=(12, 5), dpi=150)
gs = gridspec.GridSpec(1, 2, width_ratios=[1.3, 1])
ax0 = plt.subplot(gs[0])
ax1 = plt.subplot(gs[1])

# V matrix heatmap (show retained modes only)
im = ax0.imshow(V[:, :n_trunc], cmap='seismic', vmin=-1, vmax=1, aspect='auto')
ax0.set_xlabel('V-mode m')
ax0.set_xticks(range(n_trunc), [str(m+1) for m in range(n_trunc)])
ax0.set_ylabel('Normalized DOF')
yt = range(0, n_dof_sub, max(1, n_dof_sub // 8))
ax0.set_yticks(list(yt), [labels_sub[i] for i in yt])
ax0.set_title(f'V matrix — Normalized (dimensionless) DOF coefficients\n({default_dof_set}, {n_trunc} retained modes)')
fig.colorbar(im, ax=ax0, shrink=0.8)
ax0.grid(alpha=0.2)

# Singular values (1-indexed)
ax1.semilogy(np.arange(1, len(s) + 1), s, 'o-', markersize=5)
ax1.axvline(n_trunc + 0.5, color='green', ls='-', alpha=0.6, label=f'Truncation at {n_trunc}')
ax1.set_xlabel('V-mode m')
ax1.set_ylabel('σ_m')
ax1.set_title('Singular Values')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

fig.tight_layout()
pdf_save(fig)
plt.show()

In [ ]:
# Bar charts for the first 12 v-modes — Normalized (dimensionless) DOF coefficients
r = svd_results[default_dof_set]
V = r['V']
s = r['s']
labels_sub = r['labels']
n_show = min(12, V.shape[1])

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for k, ax in enumerate(axes.flat[:n_show]):
    ax.barh(range(len(labels_sub)), V[:, m], color='steelblue')
    ax.set_yticks(range(len(labels_sub)), labels_sub, fontsize=6)
    ax.set_title(f'v-mode {m+1}  (σ={s[k]:.2f})', fontsize=9)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_xlim(-1.1, 1.1)
    ax.set_xlabel('Normalized DOF coeff', fontsize=7)
    ax.tick_params(axis='x', labelsize=7)
for ax in axes.flat[n_show:]:
    ax.set_visible(False)
fig.suptitle(f'V-mode Composition — Normalized (dimensionless) DOF ({default_dof_set})', fontsize=12)
fig.tight_layout()
pdf_save(fig)

In [ ]:
# Bar charts for the first 12 v-modes — Physical DOF coefficients (V_im * n_i)
r = svd_results[default_dof_set]
V = r['V']
s = r['s']
labels_sub = r['labels']
n_sub = r['norm_sub']
n_show = min(12, V.shape[1])

# Physical V: each column is V[:, m] * n_i (physical units per unit v-mode amplitude)
V_phys = V * n_sub[:, np.newaxis]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for m, ax in enumerate(axes.flat[:n_show]):
    ax.barh(range(len(labels_sub)), V_phys[:, m], color='coral')
    ax.set_yticks(range(len(labels_sub)), labels_sub, fontsize=6)
    ax.set_title(f'v-mode {m+1}  (σ={s[m]:.2f})', fontsize=9)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel('Physical DOF (µm or deg)', fontsize=7)
    ax.grid(alpha=0.2)
for ax in axes.flat[n_show:]:
    ax.set_visible(False)
fig.suptitle(f'V-mode Composition — Physical DOF units ({default_dof_set})', fontsize=12)
fig.tight_layout()
pdf_save(fig)
plt.show()


### 3.2 Physical DOF Coefficients for V-Mode Reconstruction

The v-modes stored in nightly tables are computed as:

$$\text{vmode}_k = \sum_j V_{jk} \cdot \frac{\text{dof}_j}{n_j}$$

where $V_{jk}$ are the (dimensionless) right singular vectors and $n_i$ are the normalization weights.
To reconstruct v-modes directly from physical DOF values (in µm or degrees), the effective coefficient
for each DOF is $V_{jk} / n_i$.

The table below ranks DOF contributors by their **dimensionless weight** $|V_{jk}|$
(which reflects the true importance in the SVD), but expresses the reconstruction using
**physical coefficients** $V_{jk} / n_i$.  This avoids highlighting DOFs that only appear
large due to their normalization units (e.g. Cam_dz, M2_dz with large $n_i$).

In [ ]:
pdf_tee_start()

pdf_section('3.2 Physical DOF Coefficients for V-Mode Reconstruction')
r = svd_results[default_dof_set]
V = r['V']
labels_sub = r['labels']
n_trunc = r['n_trunc']
n_sub = r['norm_sub']

n_top = 4
print(f'Top {n_top} DOF contributors per v-mode ({default_dof_set}, {n_trunc} retained modes)')
print(f'Ranked by |V_im| (dimensionless), shown with physical coefficients V_im/n_i')
print(f'  vmode_m = sum_i (V_im / n_i) * dof_i')
print()
print(f'{"v-mode":>6s} {"sigma_m":>10s}   reconstruction from physical DOFs')
for m in range(n_trunc):
    # Rank by normalized (dimensionless) |V_im|
    order = np.argsort(-np.abs(V[:, m]))
    parts = []
    for idx in order[:n_top]:
        v_im = V[idx, m]
        if np.abs(v_im) > 0.01:
            coeff = v_im / n_sub[idx]
            parts.append(f'{coeff:+.5f}*{labels_sub[idx]}')
    print(f'{m+1:>6d} {r["s"][m]:>10.4f}   {"  ".join(parts)}')

# --- Detailed table for v-mode 5 as an example ---
print()
print('='*75)
print('Example: V-mode 5 (focus) — full coefficient table')
print('='*75)
m_example = 4  # 0-indexed
print(f'Singular value sigma_5 = {r["s"][m_example]:.4f}')
print()
print(f'{"DOF":>10s} {"V_im":>10s} {"n_i":>12s} {"V_im / n_i":>12s}')
print(f'{"-"*10:>10s} {"-"*10:>10s} {"-"*12:>12s} {"-"*12:>12s}')
order = np.argsort(-np.abs(V[:, m_example]))
for idx in order:
    v_im = V[idx, m_example]
    n_i = n_sub[idx]
    coeff = v_im / n_i
    if np.abs(v_im) > 0.005:
        print(f'{labels_sub[idx]:>10s} {v_im:>+10.4f} {n_i:>12.4f} {coeff:>+12.5f}')

pdf_tee_stop()

<a id='Zernike Patterns' ></a>
## 4. V-Mode Wavefront Signatures

The wavefront pattern of v-mode m is given by the k-th column of U, reshaped to (n_wfs, n_zernike).
This shows which Zernike terms each v-mode excites at each WFS.

In [ ]:
pdf_section('4. V-Mode Wavefront Signatures')
# Wavefront patterns at 4 WFS — use U_wfs (WFS×Zernike space, not DZ coefficients)
r = svd_results[default_dof_set]
U_wfs = r['U_wfs']
s = r['s']

n_show = min(12, U_wfs.shape[1])
fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for k, ax in enumerate(axes.flat[:n_show]):
    u_col = U_wfs[:, k].reshape(n_wfs, n_zernike)
    vmax = np.max(np.abs(u_col))
    im = ax.imshow(u_col, cmap='seismic', aspect='auto', vmin=-vmax, vmax=vmax)
    ax.set_title(f'v-mode {m+1}  (σ={s[k]:.2f})', fontsize=9)
    ax.set_yticks(range(n_wfs), [f'WFS{i}' for i in range(n_wfs)], fontsize=7)
    ax.set_xticks(range(n_zernike), [f'Z{z}' for z in zn], fontsize=5, rotation=90)
    fig.colorbar(im, ax=ax, shrink=0.7)
for ax in axes.flat[n_show:]:
    ax.set_visible(False)
fig.suptitle(f'V-Mode Wavefront Signatures at 4 WFS ({default_dof_set})', fontsize=12)
fig.tight_layout()
pdf_save(fig)

In [ ]:
# Average Zernike power per v-mode (averaged over WFS) — distinct markers per mode
r = svd_results[default_dof_set]
U_wfs = r['U_wfs']
s = r['s']
n_trunc = r['n_trunc']
n_show = min(n_trunc, U_wfs.shape[1])

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
for m in range(n_show):
    u_col = U_wfs[:, m].reshape(n_wfs, n_zernike)
    # RMS over WFS for each Zernike
    zern_rms = np.sqrt(np.mean(u_col**2, axis=0))
    mk = marker_cycle[m % len(marker_cycle)]
    ax.plot(zn, zern_rms, f'{mk}-', markersize=5, label=f'v{m+1} (σ={s[m]:.2f})')

ax.set_xlabel('Zernike (Noll)')
ax.set_ylabel('RMS of Zernike component for unit v-mode change [microns]')
ax.set_title(f'Average Zernike Power per V-Mode ({default_dof_set})')
ax.legend(fontsize=7, ncol=3)
ax.grid(alpha=0.3)
fig.tight_layout()
pdf_save(fig)

<a id='Control Equations' ></a>
## 5. Control Equations

Given a measured Zernike vector **z**, compute:
1. V-mode amplitudes: **a = U^T @ z**
2. Dimensionless DOF correction: **q = V @ Sigma^-1 @ a = Atilde^+ @ z**
3. Physical DOF correction: **delta_dof_j = n_i * q_j**

`StateEstimator` provides `get_vmodes_from_dofs` and `get_dofs_from_vmodes` for converting
between physical DOF space and v-mode space (matching nightly processing).
Helper functions for z -> q are defined in the [Code](#Code) section.

In [ ]:
pdf_tee_start()

pdf_section('5. Control Equations')
# StateEstimator DOF <-> v-mode conversion (matches nightly processing)
# r['state_estimator'].get_vmodes_from_dofs(dof_50) : physical DOFs -> v-modes
# r['state_estimator'].get_dofs_from_vmodes(vmodes)  : v-modes -> physical DOFs

# Example: M2_dz = 10 um
dof_test = np.zeros(50)
dof_test[0] = 10.0  # M2_dz
vmodes_test = r['state_estimator'].get_vmodes_from_dofs(dof_test)
n_trunc = r['n_trunc']
print(f'Test DOF: M2_dz = 10 um')
print(f'V-mode amplitudes (first {n_trunc}): {vmodes_test.round(4)}')

# Round-trip: vmodes -> DOF -> vmodes
dof_idx = r['dof_indices']
dof_recovered = r['state_estimator'].get_dofs_from_vmodes(vmodes_test)
dof_active = dof_test[dof_idx]
err = np.max(np.abs(dof_active - dof_recovered))
print(f'\nRound-trip error: {err:.2e} um')

# Verify formula: vmode_k = sum_j (V_im / n_i) * dof_j
r = svd_results[default_dof_set]
V = r['V']
n_sub = r['norm_sub']
vmodes_formula = (dof_test[dof_idx] / n_sub) @ V[:, :n_trunc]
diff = np.max(np.abs(vmodes_test - vmodes_formula))
print(f'Formula vs StateEstimator max difference: {diff:.2e}')

pdf_tee_stop()

### 5.1 Example: Wavefront from a Single V-Mode Excitation

In [ ]:
pdf_tee_start()

pdf_section('5.1 Wavefront from a Single V-Mode Excitation')
# Excite each v-mode by unit dimensionless amplitude and show the resulting Zernike pattern
r = svd_results[default_dof_set]
U = r['U']           # DZ coefficient space
U_wfs = r['U_wfs']   # WFS×Zernike space
s = r['s']
V = r['V']
Atilde_sub = r['Atilde_sub']
n_trunc = r['n_trunc']

print('Verify SVD reconstruction (DZ space): ||Ã - U @ diag(s) @ V^T|| / ||Ã|| = ',
      np.linalg.norm(Atilde_sub - U @ np.diag(s) @ V.T) / np.linalg.norm(Atilde_sub))

# Zernike vector at 4 WFS from unit v-mode m excitation: z_wfs = s_k * U_wfs_k
print(f'\nWavefront RMS at 4 WFS (µm) from unit dimensionless v-mode excitation ({n_trunc} retained modes):')
for m in range(n_trunc):
    z_from_vmode = s[m] * U_wfs[:, m]
    rms = np.sqrt(np.mean(z_from_vmode**2))
    print(f'  v-mode {m+1:2d}: σ_m={s[m]:8.4f}, wavefront RMS={rms:.4f} µm')

pdf_tee_stop()

### 5.2 Round-Trip Test: z → a → q → Δdof → z_reconstructed

Test the full control chain for two cases:
- **Case A**: 50 DOFs, all 50 v-modes (no truncation — perfect pseudoinverse)
- **Case B**: 22 DOFs, 12 retained v-modes (operational truncation)

In [ ]:
pdf_section('5.2 Round-Trip Test')
def run_roundtrip_test(dof_set_name, n_modes_use, seed=42):
    """Run a round-trip test: generate z from known q, recover q, reconstruct z."""
    r = svd_results[dof_set_name]
    V = r['V']
    s = r['s']
    Atilde_sub = r['Atilde_sub']
    n_dof = len(r['dof_indices'])
    
    # Create a test Zernike vector from a known DOF perturbation
    np.random.seed(seed)
    q_true = np.random.randn(n_dof) * 0.1
    z_test = Atilde_sub @ q_true
    
    # Recover DOFs via truncated pseudoinverse
    q_recovered = dimensionless_dof_correction(z_test, r, n_modes_use=n_modes_use)
    
    # Reconstruct Zernike from recovered DOFs
    z_reconstructed = Atilde_sub @ q_recovered
    residual = z_test - z_reconstructed
    
    dof_err = np.linalg.norm(q_true - q_recovered) / np.linalg.norm(q_true)
    z_err = np.linalg.norm(residual) / np.linalg.norm(z_test)
    
    return q_true, q_recovered, z_test, z_reconstructed, residual, dof_err, z_err

# --- Case A: 50 DOFs, all 50 v-modes ---
q_true_A, q_rec_A, z_A, z_rec_A, res_A, dof_err_A, z_err_A = run_roundtrip_test('all_50', n_modes_use=50)

# --- Case B: 22 DOFs, 12 v-modes ---
q_true_B, q_rec_B, z_B, z_rec_B, res_B, dof_err_B, z_err_B = run_roundtrip_test('standard_22', n_modes_use=12)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Case A: top row
axes[0, 0].stem(q_true_A, linefmt='b-', markerfmt='bo', basefmt='k-', label='True q')
axes[0, 0].stem(q_rec_A, linefmt='r--', markerfmt='rx', basefmt='k-', label='Recovered q')
axes[0, 0].set_xlabel('DOF index')
axes[0, 0].set_ylabel('Dimensionless DOF')
axes[0, 0].set_title(f'Case A: 50 DOFs, 50 modes — q')
axes[0, 0].legend(fontsize=7)

axes[0, 1].plot(z_A, 'b-', label='z_test')
axes[0, 1].plot(z_rec_A, 'r--', label='z_reconstructed')
axes[0, 1].set_xlabel('Zernike×WFS index')
axes[0, 1].set_ylabel('Wavefront (µm)')
axes[0, 1].set_title('z vectors')
axes[0, 1].legend(fontsize=7)

axes[0, 2].plot(res_A)
axes[0, 2].set_xlabel('Zernike×WFS index')
axes[0, 2].set_ylabel('Residual (µm)')
axes[0, 2].set_title(f'Residual (max={np.max(np.abs(res_A)):.2e})')

# Case B: bottom row
axes[1, 0].stem(q_true_B, linefmt='b-', markerfmt='bo', basefmt='k-', label='True q')
axes[1, 0].stem(q_rec_B, linefmt='r--', markerfmt='rx', basefmt='k-', label='Recovered q')
axes[1, 0].set_xlabel('DOF index')
axes[1, 0].set_ylabel('Dimensionless DOF')
axes[1, 0].set_title(f'Case B: 22 DOFs, 12 modes — q')
axes[1, 0].legend(fontsize=7)

axes[1, 1].plot(z_B, 'b-', label='z_test')
axes[1, 1].plot(z_rec_B, 'r--', label='z_reconstructed')
axes[1, 1].set_xlabel('Zernike×WFS index')
axes[1, 1].set_ylabel('Wavefront (µm)')
axes[1, 1].set_title('z vectors')
axes[1, 1].legend(fontsize=7)

axes[1, 2].plot(res_B)
axes[1, 2].set_xlabel('Zernike×WFS index')
axes[1, 2].set_ylabel('Residual (µm)')
axes[1, 2].set_title(f'Residual (max={np.max(np.abs(res_B)):.2e})')

fig.tight_layout()
pdf_save(fig)
plt.show()

print(f'Case A (50 DOFs, 50 modes): DOF error = {dof_err_A:.2e},  z error = {z_err_A:.2e}')
print(f'Case B (22 DOFs, 12 modes): DOF error = {dof_err_B:.2e},  z error = {z_err_B:.2e}')
print(f'\nCase B has nonzero error because 10 of 22 v-modes are truncated.')

<a id='Noise Amplification' ></a>
## 6. Noise Amplification and Information Content

A mode with small σ_m amplifies measurement noise: the DOF correction scales as a_k / σ_m.
If measurement noise is σ_noise in each Zernike, the DOF noise in mode m scales as σ_noise / σ_m.

This section quantifies the noise amplification (1/σ_m), the effective condition number,
and the information content vs. truncation level.

**Physical DOF noise**: The conversion from dimensionless noise $\sigma(q_j)$
to physical units is $\sigma(\Delta\text{dof}_j) = n_i \cdot \sigma(q_j)$,
where $n_i$ is the normalization weight. Translational hexapod DOFs (dx, dy)
can show very large physical noise because their normalization weights $n_i$
combine a large physical stroke range $r_j$ with high FWHM sensitivity $f_j$.
This does not mean the controller is unstable — it reflects that these DOFs
have large allowed ranges in physical units.

In [ ]:
pdf_section('6. Noise Amplification and Information Content')
r = svd_results[default_dof_set]
s = r['s']
n_trunc = r['n_trunc']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1/sigma_m: noise amplification factor (1-indexed)
axes[0].semilogy(np.arange(1, len(s) + 1), 1.0 / s, 'o-', markersize=5)
axes[0].axvline(n_trunc + 0.5, color='green', ls='-', alpha=0.6, label=f'Truncation at {n_trunc}')
axes[0].set_xlabel('V-mode m')
axes[0].set_ylabel('1/σ_m')
axes[0].set_title('Noise Amplification Factor')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Cumulative fraction of ||Ã||_F^2 captured (1-indexed)
s2_cumsum = np.cumsum(s**2)
s2_total = np.sum(s**2)
frac = s2_cumsum / s2_total
axes[1].plot(np.arange(1, len(s) + 1), frac, 'o-', markersize=5)
axes[1].axhline(0.99, color='red', ls='--', alpha=0.5, label='99%')
axes[1].axhline(0.999, color='orange', ls='--', alpha=0.5, label='99.9%')
axes[1].axvline(n_trunc + 0.5, color='green', ls='-', alpha=0.6, label=f'Truncation at {n_trunc}')
n99 = np.searchsorted(frac, 0.99) + 1
n999 = np.searchsorted(frac, 0.999) + 1
axes[1].set_xlabel('Number of modes retained')
axes[1].set_ylabel('Cumulative fraction of ||Ã||²_F')
axes[1].set_title(f'Information Content (99%→{n99} modes, 99.9%→{n999} modes)')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

# Effective condition number vs. truncation level (1-indexed)
cond = [s[0] / s[m] for m in range(len(s))]
axes[2].semilogy(range(1, len(s) + 1), cond, 'o-', markersize=5)
axes[2].axvline(n_trunc + 0.5, color='green', ls='-', alpha=0.6, label=f'Truncation at {n_trunc}')
axes[2].set_xlabel('Modes retained')
axes[2].set_ylabel('Condition number σ_1 / σ_m')
axes[2].set_title('Condition Number vs. Truncation')
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

fig.suptitle(f'Noise & Information Analysis ({default_dof_set})', fontsize=12, y=1.02)
fig.tight_layout()
pdf_save(fig)
plt.show()

### 6.1 Per-Zernike Measurement Noise from Atmospheric Turbulence

Load per-Zernike wavefront measurement noise from an atmospheric turbulence simulation (covM86.mat).
The covariance matrix gives the variance of each Zernike coefficient in nm², which we convert to µm.

In [ ]:
pdf_section('6.1 Zernike Measurement Covariance')
# Load covariance matrix from covM86.mat
# covM is 76x76: 4 WFS x 19 Zernikes (z4-z22), in nm^2
# Layout: WFS-major — rows/cols [0:19] = WFS0 z4-z22, [19:38] = WFS1, etc.
covm_data = scipy.io.loadmat('output/covM86.mat')
covM76 = covm_data['covM']
print(f'covM76 shape: {covM76.shape}, units: nm^2')

# Build full 84x84 covariance matrix for our Zernike selection (21 Zernikes x 4 WFS)
# Measurement vector layout: index = wfs_idx * n_zk_svd + zern_local_idx
n_meas = n_wfs * n_zk_svd  # 4 * 21 = 84
Cov_z = np.zeros((n_meas, n_meas))

# Map our Zernike selection to covM76 indices
# covM76 has z4-z22 (19 Zernikes), our selection has z4-z19, z22-z26 (21 Zernikes)
n_zk_covm = 19  # z4 through z22

for s1 in range(n_wfs):
    for s2 in range(n_wfs):
        for p1, z1 in enumerate(zn_svd):
            for p2, z2 in enumerate(zn_svd):
                i1 = s1 * n_zk_svd + p1  # row in our 84x84
                i2 = s2 * n_zk_svd + p2  # col in our 84x84
                
                # Both Zernikes in covM76 range (z4-z22)?
                if z1 <= 22 and z2 <= 22:
                    c1 = s1 * n_zk_covm + (z1 - 4)  # row in covM76
                    c2 = s2 * n_zk_covm + (z2 - 4)  # col in covM76
                    Cov_z[i1, i2] = covM76[c1, c2] / 1e6  # nm^2 -> um^2
                elif z1 > 22 and z2 > 22 and z1 == z2:
                    # Extra Zernikes (z23-z26): no Zernike-to-Zernike correlation,
                    # 100% WFS-to-WFS correlation per Zernike
                    default_var = (0.01)**2  # um^2 (10 nm default sigma)
                    Cov_z[i1, i2] = default_var  # same for all WFS pairs
                # else: cross-terms between covM Zernikes and extra Zernikes = 0

print(f'Full Cov_z shape: {Cov_z.shape} ({n_wfs} WFS x {n_zk_svd} Zernikes)')
print(f'Cov_z range: [{Cov_z.min():.2e}, {Cov_z.max():.2e}] um^2')

# Also keep the per-Zernike diagonal sigma for display
sigma_z_single = np.zeros(n_zk_svd)
for p, z in enumerate(zn_svd):
    # WFS0 diagonal entry
    sigma_z_single[p] = np.sqrt(Cov_z[p, p])

# Keep sigma_z_vec for backward compatibility (diagonal only)
sigma_z_vec = np.sqrt(np.diag(Cov_z))

print(f'\nPer-Zernike sigma (um, WFS0 diagonal):')
for p, z in enumerate(zn_svd):
    src = 'covM86' if z <= 22 else 'default'
    print(f'  z{z:2d}: sigma = {sigma_z_single[p]:.4f} um  ({src})')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: per-Zernike sigma
axes[0].bar(range(n_zk_svd), sigma_z_single * 1000, color='steelblue')
axes[0].set_xticks(range(n_zk_svd), [f'z{z}' for z in zn_svd], rotation=45, fontsize=8)
axes[0].set_ylabel('σ_j (nm)')
axes[0].set_title('Per-Zernike Measurement Noise (WFS0, from covM86)')
axes[0].grid(alpha=0.3, axis='y')

# Right: full covariance matrix visualization
im = axes[1].imshow(Cov_z * 1e6, aspect='equal', cmap='RdBu_r')  # show in nm^2
axes[1].set_title(f'Cov_z ({n_meas}×{n_meas}, nm²)')
axes[1].set_xlabel('Measurement index')
axes[1].set_ylabel('Measurement index')
# Mark WFS boundaries
for b in range(1, n_wfs):
    axes[1].axhline(b * n_zk_svd - 0.5, color='white', lw=0.5)
    axes[1].axvline(b * n_zk_svd - 0.5, color='white', lw=0.5)
fig.colorbar(im, ax=axes[1], shrink=0.8)

fig.tight_layout()
pdf_save(fig)
plt.show()


In [ ]:
# DOF correction noise using full Zernike covariance matrix
# Cov(q) = V @ Σ⁻¹ @ U_wfsᵀ @ Cov_z @ U_wfs @ Σ⁻¹ @ Vᵀ
# Then Cov(Δdof) = diag(n_i) @ Cov(q) @ diag(n_i)

r = svd_results[default_dof_set]
s = r['s']
V = r['V']
U_wfs = r['U_wfs']
n_trunc = r['n_trunc']
n_dof_sub = len(r['dof_indices'])
labels_sub = r['labels']
n_sub = r['norm_sub']

# Propagate full covariance through truncated pseudoinverse
# P = V[:, :n_trunc] @ diag(1/s[:n_trunc]) @ U_wfs[:, :n_trunc]^T
# Cov(q) = P @ Cov_z @ P^T
Sigma_inv_trunc = np.diag(1.0 / s[:n_trunc])
P = V[:, :n_trunc] @ Sigma_inv_trunc @ U_wfs[:, :n_trunc].T
Cov_q = P @ Cov_z @ P.T
sigma_q = np.sqrt(np.diag(Cov_q))

# Physical DOF covariance: Cov(Δdof) = diag(n_i) @ Cov(q) @ diag(n_i)
N_diag = np.diag(n_sub)
Cov_dof_phys = N_diag @ Cov_q @ N_diag
sigma_phys = np.sqrt(np.diag(Cov_dof_phys))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top left: dimensionless DOF noise
axes[0, 0].bar(range(n_dof_sub), sigma_q, color='steelblue')
axes[0, 0].set_xticks(range(n_dof_sub), labels_sub, rotation=90, fontsize=7)
axes[0, 0].set_ylabel('σ(q_i) (dimensionless)')
axes[0, 0].set_title(f'DOF Correction Noise ({default_dof_set}, {n_trunc} modes)')
axes[0, 0].grid(alpha=0.3, axis='y')

# Top right: physical DOF noise
axes[0, 1].bar(range(n_dof_sub), sigma_phys, color='coral')
axes[0, 1].set_xticks(range(n_dof_sub), labels_sub, rotation=90, fontsize=7)
axes[0, 1].set_ylabel('σ(Δdof_i) (physical units)')
axes[0, 1].set_title(f'Physical DOF Noise ({default_dof_set}, {n_trunc} modes)')
axes[0, 1].grid(alpha=0.3, axis='y')

# Bottom left: dimensionless DOF covariance matrix
vmax_q = np.max(np.abs(Cov_q))
im0 = axes[1, 0].imshow(Cov_q, aspect='equal', cmap='RdBu_r', vmin=-vmax_q, vmax=vmax_q)
axes[1, 0].set_title('Cov(q) — dimensionless DOF covariance')
axes[1, 0].set_xticks(range(0, n_dof_sub, 2), [labels_sub[i] for i in range(0, n_dof_sub, 2)],
                       rotation=90, fontsize=6)
axes[1, 0].set_yticks(range(0, n_dof_sub, 2), [labels_sub[i] for i in range(0, n_dof_sub, 2)],
                       fontsize=6)
fig.colorbar(im0, ax=axes[1, 0], shrink=0.8)

# Bottom right: physical DOF covariance matrix
vmax_phys = np.max(np.abs(Cov_dof_phys))
im1 = axes[1, 1].imshow(Cov_dof_phys, aspect='equal', cmap='RdBu_r', vmin=-vmax_phys, vmax=vmax_phys)
axes[1, 1].set_title('Cov(Δdof) — physical DOF covariance')
axes[1, 1].set_xticks(range(0, n_dof_sub, 2), [labels_sub[i] for i in range(0, n_dof_sub, 2)],
                       rotation=90, fontsize=6)
axes[1, 1].set_yticks(range(0, n_dof_sub, 2), [labels_sub[i] for i in range(0, n_dof_sub, 2)],
                       fontsize=6)
fig.colorbar(im1, ax=axes[1, 1], shrink=0.8)

fig.suptitle(f'DOF Noise from Zernike Covariance (covM86, {default_dof_set})', fontsize=13)
fig.tight_layout()
pdf_save(fig)
plt.show()


### 6.2 Effect of Truncation on Noise vs. Reconstruction Error

In [ ]:
pdf_section('6.2 Truncation vs. Noise Tradeoff')
r = svd_results[default_dof_set]
s = r['s']
U_wfs = r['U_wfs']
V = r['V']
n_modes_total = len(s)

# For each truncation level, compute:
# 1. Reconstruction error: ||Ã - Ã_trunc||_F / ||Ã||_F
# 2. Total DOF noise using full Zernike covariance
recon_error = []
total_dof_noise = []

for n_keep in range(1, n_modes_total + 1):
    # Truncated reconstruction error
    residual_frac = np.sqrt(np.sum(s[n_keep:]**2)) / np.sqrt(np.sum(s**2))
    recon_error.append(residual_frac)
    
    # DOF noise from retained modes using full covariance
    Sigma_inv_trunc = np.diag(1.0 / s[:n_keep])
    P_trunc = V[:, :n_keep] @ Sigma_inv_trunc @ U_wfs[:, :n_keep].T
    Cov_q_trunc = P_trunc @ Cov_z @ P_trunc.T
    noise_var = np.trace(Cov_q_trunc)
    total_dof_noise.append(np.sqrt(noise_var))

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.semilogy(range(1, n_modes_total + 1), recon_error, 'bo-', markersize=5, label='Reconstruction error')
ax2.semilogy(range(1, n_modes_total + 1), total_dof_noise, 'rs-', markersize=5, label='DOF noise')

ax1.set_xlabel('V-modes retained')
ax1.set_ylabel('Fractional reconstruction error', color='blue')
ax2.set_ylabel('Total DOF noise (dimensionless)', color='red')
ax1.set_title(f'Truncation Tradeoff ({default_dof_set}, full Cov_z from covM86)')
ax1.grid(alpha=0.3)

lines1, lab1 = ax1.get_legend_handles_labels()
lines2, lab2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lab1 + lab2, loc='center right')

fig.tight_layout()
pdf_save(fig)
plt.show()


<a id='Control Gain' ></a>
## 7. Control Gain Analysis

The AOS control loop operates as a discrete-time feedback system:

1. **Measure:** Read out wavefront Zernike coefficients from the 4 corner WFS (84-element vector **z**)
2. **Project:** Compute v-mode amplitudes **a** = Uᵀ **z** — this decomposes the measured wavefront error into the independent correction modes
3. **Correct:** Compute the DOF correction **Δq** = −K_P × Σ⁻¹ **a** — the pseudoinverse (Σ⁻¹) converts wavefront amplitudes to dimensionless DOF motions, and K_P is the proportional gain that controls how aggressively we correct
4. **Apply:** Send physical corrections Δdof_j = n_i × Δq_j to the hexapod and mirror actuators. Due to the readout/computation pipeline, there is a multi-step delay before the correction takes effect.

**Why does σ_m matter for gain?**
The pseudoinverse divides each v-mode amplitude by its singular value σ_m. When the correction is applied, the optics respond with sensitivity σ_m. So the net effect of a full correction (K_P = 1) would be: the wavefront change equals σ_m × (a_k / σ_m) = a_k — a perfect one-step correction. The gain K_P < 1 intentionally reduces this to ensure stability in the presence of delay.

**Uniform gain K_P:**
The effective closed-loop gain for v-mode m is K_P (independent of σ_m for the uniform case). However, in practice the *sensitivity* to noise and model errors varies by mode — weak modes (small σ_m) amplify noise more because the pseudoinverse divides by a small number. The correction step for mode m is K_P × a_k / σ_m, so noise in a_k gets amplified by 1/σ_m before being multiplied by K_P.

**Per-mode gain scheduling:** Setting K_P(m) = gain_factor / σ_m gives each mode an effective correction fraction of gain_factor × a_k / σ_m × σ_m = gain_factor × a_k. But more importantly, it reduces the gain for noise-amplifying weak modes.

**Plots below:**
- **Left:** Effective gain K_P × σ_m per mode under uniform K_P — shows the spread across modes
- **Center:** The per-mode K_P(m) schedule — weaker modes get larger K_P to compensate
- **Right:** Convergence rate comparison: fraction of initial error remaining vs. image number

In [ ]:
pdf_section('7. Control Gain Analysis')
r = svd_results[default_dof_set]
s = r['s']
n_trunc = r['n_trunc']

# --- Uniform gain: all modes use the same K_P ---
# Effective gain per mode = K_P * sigma_m
# Large-sigma modes have high effective gain, small-sigma modes have low effective gain
K_P_uniform = 0.2
effective_gain_uniform = K_P_uniform * s[:n_trunc]

# --- Per-mode gain scheduling: K_P(m) = gain_factor / sigma_m ---
# This equalizes the effective gain to gain_factor for all modes
# Weak modes get a larger K_P to compensate for their small sigma_m
gain_factor = 0.35
K_P_per_mode = gain_factor / s[:n_trunc]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Left: effective gain per mode under uniform K_P
axes[0].bar(np.arange(1, n_trunc + 1), effective_gain_uniform, color='steelblue')
axes[0].axhline(1.0, color='red', ls='--', label='Stability limit')
axes[0].set_xlabel('V-mode m')
axes[0].set_ylabel('Effective gain K_P × σ_m')
axes[0].set_title(f'Uniform K_P = {K_P_uniform}')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3, axis='y')

# Center: per-mode K_P values (log scale since they span a wide range)
axes[1].semilogy(np.arange(1, n_trunc + 1), K_P_per_mode, 'o-', markersize=5)
axes[1].axhline(K_P_uniform, color='gray', ls='--', alpha=0.5, label=f'Uniform K_P={K_P_uniform}')
axes[1].set_xlabel('V-mode m')
axes[1].set_ylabel('K_P(m) = gain_factor / σ_m')
axes[1].set_title(f'Per-mode Gain Schedule (factor={gain_factor})')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

# Right: convergence comparison — fraction of error remaining after N images
# With uniform K_P and no delay: a_m[n+1] = (1 - K_P) * a_m[n]
# With per-mode scheduling: a_m[n+1] = (1 - gain_factor) * a_m[n]
N = np.arange(1, 31)
pole_uniform = 1.0 - K_P_uniform
residual = pole_uniform**N
axes[2].plot(N, residual, '-', label=f'Uniform K_P={K_P_uniform}', alpha=0.7)

pole_sched = 1.0 - gain_factor
residual_sched = pole_sched**N
axes[2].plot(N, residual_sched, '--', label=f'Scheduled (factor={gain_factor})', alpha=0.7)

axes[2].set_xlabel('Images (N)')
axes[2].set_ylabel('Fraction remaining')
axes[2].set_title('Convergence (no delay)')
axes[2].set_yscale('log')
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

fig.tight_layout()
pdf_save(fig)
plt.show()

### 7.1 Per-Mode Convergence with 2-Step Delay

The simulation models each v-mode independently as a discrete control loop:

$$a_m[n+1] = a_m[n] - K_P \cdot a_m[n - d]$$

where $a_m[n]$ is the residual v-mode amplitude at image $n$, $K_P$ is the
proportional gain, and $d$ is the delay (number of images between measurement
and correction). Each mode starts at $a_m[0] = 1$ (unit initial error) and
converges toward zero. The delay arises because wavefront sensing and DOF
correction take time: a $d$-step delay means the correction applied before
image $n+1$ is based on the measurement from image $n - d + 1$.

**Uniform gain**: all modes use the same $K_P$. Modes with large $\sigma_m$
converge quickly but the system is limited by stability of the weakest mode.

**Per-mode scheduling**: $K_P(m) = \text{gain\_factor} / \sigma_m$, equalizing
the effective loop gain across all modes. This helps weak modes converge faster
but can become unstable with longer delays because the large $K_P$ values for
weak modes interact badly with the delay.

Note: this is a single-mode, deterministic simulation — no measurement noise,
no cross-mode coupling. It shows the *convergence dynamics* of the feedback
loop, not the steady-state residual (which depends on noise).

In [ ]:
pdf_section('7.1 Per-Mode Convergence (2-Step Delay)')
# Simulate the discrete control loop with 2-step delay for each v-mode independently
# a_m[n+1] = a_m[n] - K_P * a_m[n - delay]

r = svd_results[default_dof_set]
s = r['s']
n_trunc = r['n_trunc']
n_images = 40
delay = 2  # correction from image n applied before image n+3

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Uniform gain
K_P = 0.2
for m in range(min(6, n_trunc)):
    x = np.zeros(n_images + delay + 1)
    x[0] = 1.0
    for n in range(n_images):
        if n + 1 <= delay:
            x[n + 1] = x[n]
        else:
            x[n + 1] = x[n] - K_P * x[n - delay]
    axes[0].plot(range(n_images + 1), x[:n_images + 1], '.-', markersize=3,
                 label=f'v-mode {m+1} (σ={s[m]:.2f})')

axes[0].set_xlabel('Image number')
axes[0].set_ylabel('Fraction of initial error')
axes[0].set_title(f'Uniform K_P={K_P}, {delay}-step delay')
axes[0].legend(fontsize=7)
axes[0].grid(alpha=0.3)
axes[0].axhline(0, color='k', lw=0.5)

# Per-mode gain scheduling: a_m[n+1] = a_m[n] - gain_factor * a_m[n-delay]
gain_factor = 0.35
for m in range(min(6, n_trunc)):
    x = np.zeros(n_images + delay + 1)
    x[0] = 1.0
    for n in range(n_images):
        if n + 1 <= delay:
            x[n + 1] = x[n]
        else:
            x[n + 1] = x[n] - gain_factor * x[n - delay]
    axes[1].plot(range(n_images + 1), x[:n_images + 1], '.-', markersize=3,
                 label=f'v-mode {m+1} (σ={s[m]:.2f})')

axes[1].set_xlabel('Image number')
axes[1].set_ylabel('Fraction of initial error')
axes[1].set_title(f'Per-mode scheduling, gain_factor={gain_factor}, {delay}-step delay')
axes[1].legend(fontsize=7)
axes[1].grid(alpha=0.3)
axes[1].axhline(0, color='k', lw=0.5)

fig.tight_layout()
pdf_save(fig)
plt.show()

### 7.2 Per-Mode Convergence with 1-Step Delay

With a 1-step delay (correction from image n applied before image n+2), the system
converges faster than the 2-step case. This is the best-case scenario for the AOS loop.

In [ ]:
pdf_section('7.2 Per-Mode Convergence (1-Step Delay)')
# Simulate the discrete control loop with 1-step delay for each v-mode independently
# a_m[n+1] = a_m[n] - K_P * a_m[n - delay]

r = svd_results[default_dof_set]
s = r['s']
n_trunc = r['n_trunc']
n_images = 40
delay = 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Uniform gain
K_P = 0.2
for m in range(min(6, n_trunc)):
    x = np.zeros(n_images + delay + 1)
    x[0] = 1.0
    for n in range(n_images):
        if n + 1 <= delay:
            x[n + 1] = x[n]
        else:
            x[n + 1] = x[n] - K_P * x[n - delay]
    axes[0].plot(range(n_images + 1), x[:n_images + 1], '.-', markersize=3,
                 label=f'v-mode {m+1} (σ={s[m]:.2f})')

axes[0].set_xlabel('Image number')
axes[0].set_ylabel('Fraction of initial error')
axes[0].set_title(f'Uniform K_P={K_P}, {delay}-step delay')
axes[0].legend(fontsize=7)
axes[0].grid(alpha=0.3)
axes[0].axhline(0, color='k', lw=0.5)

# Per-mode gain scheduling
gain_factor = 0.35
for m in range(min(6, n_trunc)):
    x = np.zeros(n_images + delay + 1)
    x[0] = 1.0
    for n in range(n_images):
        if n + 1 <= delay:
            x[n + 1] = x[n]
        else:
            x[n + 1] = x[n] - gain_factor * x[n - delay]
    axes[1].plot(range(n_images + 1), x[:n_images + 1], '.-', markersize=3,
                 label=f'v-mode {m+1} (σ={s[m]:.2f})')

axes[1].set_xlabel('Image number')
axes[1].set_ylabel('Fraction of initial error')
axes[1].set_title(f'Per-mode scheduling, gain_factor={gain_factor}, {delay}-step delay')
axes[1].legend(fontsize=7)
axes[1].grid(alpha=0.3)
axes[1].axhline(0, color='k', lw=0.5)

fig.tight_layout()
pdf_save(fig)
plt.show()

### 7.3 Per-Mode Convergence with 3-Step Delay

With a 3-step delay (correction from image n not applied until image n+4), the system
converges more slowly and is more prone to oscillation. This represents a pessimistic
scenario for the AOS loop.

In [ ]:
pdf_section('7.3 Per-Mode Convergence (3-Step Delay)')
# Simulate the discrete control loop with 3-step delay for each v-mode independently
# Standard mode: a_m[n+1] = a_m[n] - K_P * a_m[n - delay]
# Skip mode: only apply corrections every `delay` images
#   (measure on image 1, apply before image 1+delay, skip images 2..delay)

r = svd_results[default_dof_set]
s = r['s']
n_trunc = r['n_trunc']
n_images = 40
delay = 3

n_cols = 3 if use_skip_mode else 2
fig, axes = plt.subplots(1, n_cols, figsize=(7 * n_cols, 5))

# Left: Uniform gain, every-image corrections
K_P = 0.2
for m in range(min(6, n_trunc)):
    x = np.zeros(n_images + delay + 1)
    x[0] = 1.0
    for n in range(n_images):
        if n + 1 <= delay:
            x[n + 1] = x[n]
        else:
            x[n + 1] = x[n] - K_P * x[n - delay]
    axes[0].plot(range(n_images + 1), x[:n_images + 1], '.-', markersize=3,
                 label=f'v{m+1} (σ={s[m]:.2f})')

axes[0].set_xlabel('Image number')
axes[0].set_ylabel('Fraction of initial error')
axes[0].set_title(f'Uniform K_P={K_P}, {delay}-step delay')
axes[0].legend(fontsize=7)
axes[0].grid(alpha=0.3)
axes[0].axhline(0, color='k', lw=0.5)

# Center: Per-mode gain scheduling, every-image corrections
gain_factor = 0.35
for m in range(min(6, n_trunc)):
    x = np.zeros(n_images + delay + 1)
    x[0] = 1.0
    for n in range(n_images):
        if n + 1 <= delay:
            x[n + 1] = x[n]
        else:
            x[n + 1] = x[n] - gain_factor * x[n - delay]
    axes[1].plot(range(n_images + 1), x[:n_images + 1], '.-', markersize=3,
                 label=f'v{m+1} (σ={s[m]:.2f})')

axes[1].set_xlabel('Image number')
axes[1].set_ylabel('Fraction of initial error')
axes[1].set_title(f'Per-mode, factor={gain_factor}, {delay}-step delay')
axes[1].legend(fontsize=7)
axes[1].grid(alpha=0.3)
axes[1].axhline(0, color='k', lw=0.5)

if use_skip_mode:
    # Right: Skip mode — apply corrections only every `delay` images
    # With delay=3: measure at image 0, apply correction before image 3,
    # then measure at image 3, apply before image 6, etc.
    # Between corrections, x is constant (no new corrections applied).
    # At correction steps: x[n] = x[n-1] - gain * x[n - delay]
    # but only at n where n % delay == 0

    # Uniform gain with skip mode
    for m in range(min(6, n_trunc)):
        x = np.zeros(n_images + delay + 1)
        x[0] = 1.0
        for n in range(n_images):
            if n + 1 <= delay:
                x[n + 1] = x[n]
            elif (n + 1) % delay == 0:
                # Apply correction from measurement `delay` steps ago
                x[n + 1] = x[n] - K_P * x[n + 1 - delay]
            else:
                x[n + 1] = x[n]  # no correction applied
        axes[2].plot(range(n_images + 1), x[:n_images + 1], '.-', markersize=3,
                     label=f'v{m+1} (σ={s[m]:.2f})')

    axes[2].set_xlabel('Image number')
    axes[2].set_ylabel('Fraction of initial error')
    axes[2].set_title(f'Skip mode: uniform K_P={K_P}, correct every {delay} images')
    axes[2].legend(fontsize=7)
    axes[2].grid(alpha=0.3)
    axes[2].axhline(0, color='k', lw=0.5)

fig.tight_layout()
pdf_save(fig)
plt.show()


<a id='DOF Set Comparison' ></a>
## 8. Comparison Across DOF Sets

Compare SVD properties for different DOF subsets (10, 22, 30, 50 DOFs).

In [ ]:
pdf_section('8. Comparison Across DOF Sets')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, r in svd_results.items():
    s = r['s']
    n = len(s)
    axes[0].semilogy(np.arange(1, n + 1), s, '.-', markersize=5, label=f'{name} ({n} DOFs)')
    
    s2_cumsum = np.cumsum(s**2)
    axes[1].plot(np.arange(1, n + 1), s2_cumsum / s2_cumsum[-1], '.-', markersize=5,
                 label=f'{name} ({n} DOFs)')

axes[0].set_xlabel('V-mode m')
axes[0].set_ylabel('σ_m')
axes[0].set_title('Singular Value Spectra')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].axhline(0.99, color='red', ls='--', alpha=0.5)
axes[1].set_xlabel('Modes retained')
axes[1].set_ylabel('Cumulative fraction of ||Ã||²')
axes[1].set_title('Information Content')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

fig.tight_layout()
pdf_save(fig)
plt.show()

In [ ]:
# V-matrix heatmaps for all DOF sets (1-indexed)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, (name, r) in zip(axes.flat, svd_results.items()):
    V = r['V']
    labels_sub = r['labels']
    n_dof = len(labels_sub)
    n_modes = V.shape[1]
    im = ax.imshow(V, cmap='seismic', vmin=-1, vmax=1, aspect='auto')
    ax.set_title(f'{name} ({n_dof} DOFs, {n_modes} modes)', fontsize=10)
    ax.set_xlabel('V-mode m')
    xt = np.arange(0, n_modes, max(1, n_modes // 10))
    ax.set_xticks(xt, [str(x + 1) for x in xt])
    yt = range(0, n_dof, max(1, n_dof // 6))
    ax.set_yticks(list(yt), [labels_sub[i] for i in yt], fontsize=7)
    fig.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('V Matrices — Normalized (dimensionless) DOF Composition of V-Modes', fontsize=13)
fig.tight_layout()
pdf_save(fig)
plt.show()

<a id='DOF Set Comparisons' ></a>
## 9. DOF Set Comparisons

In [ ]:
pdf_tee_start()

pdf_section('9. DOF Set Comparisons')
print(f'{"DOF Set":>15s} {"n_DOF":>6s} {"n_trunc":>7s} {"σ_max":>10s} {"σ_min":>10s} '
      f'{"Cond #":>10s} {"99% modes":>10s} {"99.9% modes":>12s}')
print('-' * 90)

for name, r in svd_results.items():
    s = r['s']
    n_trunc = r['n_trunc']
    frac = np.cumsum(s**2) / np.sum(s**2)
    n99 = np.searchsorted(frac, 0.99) + 1
    n999 = np.searchsorted(frac, 0.999) + 1
    print(f'{name:>15s} {len(r["dof_indices"]):>6d} {n_trunc:>7d} {s[0]:>10.3f} {s[-1]:>10.2e} '
          f'{s[0]/s[-1]:>10.1f} {n99:>10d} {n999:>12d}')

pdf_tee_stop()

<a id='Double-Zernike' ></a>
## 10. Double-Zernike V-Mode Decomposition

The double-Zernike basis describes wavefront patterns that are Zernike polynomial j over the pupil 
and Zernike polynomial k over the focal plane (Noll convention).

The focal plane positions are the 4 WFS corners. We use:
- **k_fp = 1** (constant over focal plane): field-averaged aberration
- **k_fp = 2** (x-tilt over focal plane): x-gradient of aberration
- **k_fp = 3** (y-tilt over focal plane): y-gradient of aberration

For each double-Zernike term (j, k_fp), we find:
1. Which v-modes are needed to produce this pattern
2. Whether the correction is "clean" (no crosstalk to other double-Zernike terms)

Analysis range: j = 4–11 for k_fp = 1; j = 4–8 for k_fp = 2, 3.

In [ ]:
pdf_tee_start()

pdf_section('10. Double-Zernike V-Mode Decomposition')
# --- Build the double-Zernike basis vectors ---
# Measurement vector layout matches StateEstimator's U:
#   index i = wfs_idx * n_zk_svd + zern_local_idx
# where zern_local_idx indexes into zn_svd array (0..n_zk_svd-1)

# Focal plane Zernike polynomials evaluated at 4 WFS positions
# WFS positions (field angles in degrees)
wfs_xy = np.array(field_angles)  # shape (4, 2)
print('WFS focal plane positions (deg):')
for i, (x, y) in enumerate(wfs_xy):
    print(f'  WFS{i}: ({x:.4f}, {y:.4f})')

# Normalize positions to unit circle radius = max distance from center
R_fp = np.max(np.sqrt(wfs_xy[:, 0]**2 + wfs_xy[:, 1]**2))
wfs_x_norm = wfs_xy[:, 0] / R_fp
wfs_y_norm = wfs_xy[:, 1] / R_fp
print(f'\nNormalization radius R_fp = {R_fp:.4f} deg')
print(f'Normalized WFS positions: x = {wfs_x_norm}, y = {wfs_y_norm}')

# Focal plane Zernike values (Noll convention):
# k_fp=1: Z_1 = 1 (piston/constant)
# k_fp=2: Z_2 = 2*x (tilt-x)
# k_fp=3: Z_3 = 2*y (tilt-y)
fp_zernike = {
    1: np.ones(n_wfs),                  # constant
    2: 2.0 * wfs_x_norm,               # x-tilt
    3: 2.0 * wfs_y_norm,               # y-tilt
}

fp_zernike_names = {1: 'const', 2: 'x-tilt', 3: 'y-tilt'}

print('\nFocal plane Zernike values at WFS positions:')
for kfp, vals in fp_zernike.items():
    print(f'  k_fp={kfp} ({fp_zernike_names[kfp]}): {vals}')

# Build double-Zernike basis vectors
# dz_vector(j, k_fp) has length n_wfs * n_zk_svd (matching U dimensions)
# Component at index (s * n_zk_svd + p) = delta(zn_svd[p], j) * Z_{k_fp}(wfs_s)
def make_dz_vector(j_noll, k_fp):
    """DZ basis vector (pupil j × focal k_fp) via ofc_svd."""
    return osv.make_dz_basis_vector(j_noll, k_fp, zn_svd,
                                    fp_zernike, n_wfs, n_zk_svd)

# Define the double-Zernike terms to analyze
dz_terms = []
# j = 4-15 for k_fp = 1
for j in range(4, 16):
    dz_terms.append((j, 1))
# j = 4-8 for k_fp = 2 and 3
for j in range(4, 9):
    dz_terms.append((j, 2))
for j in range(4, 9):
    dz_terms.append((j, 3))

print(f'\n{len(dz_terms)} double-Zernike terms to analyze:')
for j, kfp in dz_terms:
    print(f'  (j={j}, k_fp={kfp}) = z{j} x {fp_zernike_names[kfp]}')

# Build all DZ basis vectors
dz_vectors = {}
for j, kfp in dz_terms:
    vec = make_dz_vector(j, kfp)
    if vec is not None:
        dz_vectors[(j, kfp)] = vec
    else:
        print(f'  WARNING: z{j} not in Zernike selection, skipping (j={j}, k_fp={kfp})')

n_dz = len(dz_vectors)
print(f'\nBuilt {n_dz} double-Zernike basis vectors (length {n_wfs * n_zk_svd})')

pdf_tee_stop()

### 10.1 V-Mode Projections of Double-Zernike Terms

For each double-Zernike term (j, k_fp), project onto the U basis to find v-mode amplitudes:
**a_m = U^T_k @ dz_vector(j, k_fp)**

Then reconstruct and decompose back into double-Zernike components to check for crosstalk.

In [ ]:
pdf_tee_start()

pdf_section('10.1 V-Mode Projections of DZ Terms')
# --- Project each DZ term onto v-modes and check crosstalk ---
r = svd_results[default_dof_set]
s = r['s']
n_trunc = r['n_trunc']
U_wfs_dz = r['U_wfs']  # preserves StateEstimator v-mode ordering

# QR factorization of U_wfs[:, :n_trunc] for orthogonal projection
# onto the column space (for achievability), while keeping v-mode numbering
Q_trunc, _ = np.linalg.qr(U_wfs_dz[:, :n_trunc])

# Function to decompose a measurement vector into double-Zernike coefficients
from ofc_svd import decompose_into_dz


# For each DZ term, compute:
# 1. v-mode amplitudes a_k (via lstsq on U_wfs — preserves mode ordering)
# 2. Fraction of DZ vector in the column space of U_wfs[:, :n_trunc] (achievability)
# 3. Crosstalk: which other DZ terms are excited

dz_term_list = list(dz_vectors.keys())
n_dz = len(dz_term_list)

print(f'Double-Zernike V-Mode Analysis ({default_dof_set}, {n_trunc} retained modes)')
print('=' * 100)

# Store results for later plotting
dz_vmode_amps = {}   # (j, kfp) -> array of v-mode amplitudes
dz_achievable = {}   # (j, kfp) -> fraction achievable
dz_crosstalk = {}    # (j, kfp) -> dict of crosstalk coefficients

for j, kfp in dz_term_list:
    dz_vec = dz_vectors[(j, kfp)]
    
    # V-mode amplitudes via least-squares (preserves StateEstimator ordering)
    a, _, _, _ = np.linalg.lstsq(U_wfs_dz[:, :n_trunc], dz_vec, rcond=None)
    dz_vmode_amps[(j, kfp)] = a
    
    # Reconstruct from retained v-modes
    z_reconstructed = U_wfs_dz[:, :n_trunc] @ a
    
    # Achievability: orthogonal projection via QR
    z_proj = Q_trunc @ (Q_trunc.T @ dz_vec)
    frac_achieved = np.linalg.norm(z_proj) / np.linalg.norm(dz_vec) if np.linalg.norm(dz_vec) > 0 else 0
    dz_achievable[(j, kfp)] = frac_achieved
    
    # Decompose the reconstructed vector into DZ components
    recon_dz_coeffs = decompose_into_dz(z_reconstructed, dz_vectors)
    dz_crosstalk[(j, kfp)] = recon_dz_coeffs
    
    # Print significant v-mode contributions (1-indexed)
    sig_modes = np.where(np.abs(a) > 0.01 * np.max(np.abs(a)))[0]
    mode_str = ', '.join([f'v{m+1}({a[m]:+.3f})' for m in sig_modes[:6]])
    
    # Check crosstalk
    self_coeff = recon_dz_coeffs.get((j, kfp), 0.0)
    xtalk_terms = [(jj, kk, c) for (jj, kk), c in recon_dz_coeffs.items() 
                   if (jj, kk) != (j, kfp) and abs(c) > 0.01 * abs(self_coeff) and abs(c) > 1e-6]
    
    xtalk_str = 'CLEAN' if len(xtalk_terms) == 0 else ', '.join(
        [f'z{jj}\u00d7{fp_zernike_names[kk]}({c:.3f})' for jj, kk, c in xtalk_terms[:4]])
    
    print(f'z{j}\u00d7{fp_zernike_names[kfp]:>7s}: achievable={frac_achieved:.4f}  '
          f'modes=[{mode_str}]  crosstalk: {xtalk_str}')

pdf_tee_stop()

### 10.2 V-Mode Amplitude Heatmap for Double-Zernike Terms

Each row is a double-Zernike term (j, k_fp), each column is a v-mode.
Color shows the amplitude a_m = U^T_k @ dz_vector(j, k_fp).

In [ ]:
pdf_section('10.2 V-Mode Amplitude Heatmap for DZ Terms')
# Build the DZ → v-mode amplitude matrix
n_trunc = r['n_trunc']
amp_matrix = np.zeros((n_dz, n_trunc))
dz_labels = []
for i, (j, kfp) in enumerate(dz_term_list):
    amp_matrix[i, :] = dz_vmode_amps[(j, kfp)]
    dz_labels.append(f'z{j}×{fp_zernike_names[kfp]}')

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Heatmap of v-mode amplitudes
vmax = np.max(np.abs(amp_matrix))
im = axes[0].imshow(amp_matrix, cmap='seismic', vmin=-vmax, vmax=vmax, aspect='auto')
axes[0].set_xlabel('V-mode m')
axes[0].set_xticks(range(n_trunc), [str(m+1) for m in range(n_trunc)])
axes[0].set_ylabel('Double-Zernike term (j, k_fp)')
axes[0].set_yticks(range(n_dz), dz_labels, fontsize=7)
axes[0].set_title(f'V-mode amplitudes for each DZ term\n({default_dof_set}, {n_trunc} modes)')
fig.colorbar(im, ax=axes[0], shrink=0.8)

# Achievability bar chart
achiev = [dz_achievable[(j, kfp)] for j, kfp in dz_term_list]
bar_colors = ['green' if a > 0.99 else 'orange' if a > 0.9 else 'red' for a in achiev]
axes[1].barh(range(n_dz), achiev, color=bar_colors)
axes[1].set_yticks(range(n_dz), dz_labels, fontsize=7)
axes[1].set_xlabel('Achievability (||z_recon|| / ||z_target||)')
axes[1].set_title(f'Fraction of DZ term achievable with {n_trunc} v-modes')
axes[1].axvline(1.0, color='k', ls='--', alpha=0.3)
axes[1].set_xlim(0, 1.05)
axes[1].grid(alpha=0.3, axis='x')

fig.tight_layout()
pdf_save(fig)
plt.show()

In [ ]:
# V-mode amplitude heatmaps for DZ terms across ALL DOF/truncation sets
# For each DOF set, project DZ terms using that set's U_wfs (preserving v-mode ordering)

for dof_name in dof_sets.keys():
    r_i = svd_results[dof_name]
    n_trunc_i = r_i['n_trunc']
    U_wfs_i = r_i['U_wfs']
    Q_i, _ = np.linalg.qr(U_wfs_i[:, :n_trunc_i])

    # Project each DZ term onto this set's v-modes
    amp_matrix_i = np.zeros((n_dz, n_trunc_i))
    dz_labels_i = []
    achiev_i = []
    for di, (j, kfp) in enumerate(dz_term_list):
        dz_vec = dz_vectors[(j, kfp)]
        a_i, _, _, _ = np.linalg.lstsq(U_wfs_i[:, :n_trunc_i], dz_vec, rcond=None)
        amp_matrix_i[di, :] = a_i
        dz_labels_i.append(f'z{j}\u00d7{fp_zernike_names[kfp]}')

        z_proj = Q_i @ (Q_i.T @ dz_vec)
        frac = np.linalg.norm(z_proj) / np.linalg.norm(dz_vec) if np.linalg.norm(dz_vec) > 0 else 0
        achiev_i.append(frac)

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    vmax = np.max(np.abs(amp_matrix_i))
    im = axes[0].imshow(amp_matrix_i, cmap='seismic', vmin=-vmax, vmax=vmax, aspect='auto')
    axes[0].set_xlabel('V-mode m')
    axes[0].set_xticks(range(n_trunc_i), [str(m+1) for m in range(n_trunc_i)])
    axes[0].set_ylabel('Double-Zernike term')
    axes[0].set_yticks(range(n_dz), dz_labels_i, fontsize=7)
    axes[0].set_title(f'V-mode amplitudes ({dof_name}, {n_trunc_i} modes)')
    fig.colorbar(im, ax=axes[0], shrink=0.8)

    bar_colors = ['green' if a > 0.99 else 'orange' if a > 0.9 else 'red' for a in achiev_i]
    axes[1].barh(range(n_dz), achiev_i, color=bar_colors)
    axes[1].set_yticks(range(n_dz), dz_labels_i, fontsize=7)
    axes[1].set_xlabel('Achievability')
    axes[1].set_title(f'Fraction achievable ({dof_name}, {n_trunc_i} modes)')
    axes[1].axvline(1.0, color='k', ls='--', alpha=0.3)
    axes[1].set_xlim(0, 1.05)
    axes[1].grid(alpha=0.3, axis='x')

    fig.tight_layout()
    pdf_save(fig)
    plt.show()


### 10.3 Crosstalk Matrix

The crosstalk matrix C(i, j) shows: when we try to correct DZ term i using the retained v-modes,
how much of DZ term j gets excited. Diagonal = self-correction (should be ~1 for achievable terms).
Off-diagonal = crosstalk (should be ~0 for clean corrections).

In [ ]:
pdf_section('10.3 Crosstalk Matrix')
# Build the crosstalk matrix
# C[i, j] = when correcting DZ term i, how much of DZ term j appears
n_trunc = r['n_trunc']
crosstalk_matrix = np.zeros((n_dz, n_dz))

for i, (j_i, kfp_i) in enumerate(dz_term_list):
    xtalk = dz_crosstalk[(j_i, kfp_i)]
    for ii, (j_ii, kfp_ii) in enumerate(dz_term_list):
        crosstalk_matrix[i, ii] = xtalk.get((j_ii, kfp_ii), 0.0)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Full crosstalk matrix
vmax = np.max(np.abs(crosstalk_matrix))
im = axes[0].imshow(crosstalk_matrix, cmap='seismic', vmin=-vmax, vmax=vmax, aspect='auto')
axes[0].set_xlabel('Target DZ term (excited)')
axes[0].set_ylabel('Input DZ term (corrected)')
axes[0].set_xticks(range(n_dz), dz_labels, rotation=90, fontsize=6)
axes[0].set_yticks(range(n_dz), dz_labels, fontsize=6)
axes[0].set_title(f'Crosstalk Matrix ({default_dof_set}, {n_trunc} modes)')
fig.colorbar(im, ax=axes[0], shrink=0.8)

# Off-diagonal crosstalk magnitude (log scale)
offdiag = crosstalk_matrix.copy()
np.fill_diagonal(offdiag, 0)
vmax_off = np.max(np.abs(offdiag))
if vmax_off > 0:
    im2 = axes[1].imshow(np.abs(offdiag), norm=colors.LogNorm(vmin=1e-6, vmax=max(vmax_off, 1e-5)),
                          aspect='auto', cmap='hot_r')
else:
    im2 = axes[1].imshow(np.abs(offdiag), aspect='auto', cmap='hot_r')
axes[1].set_xlabel('Target DZ term')
axes[1].set_ylabel('Input DZ term')
axes[1].set_xticks(range(n_dz), dz_labels, rotation=90, fontsize=6)
axes[1].set_yticks(range(n_dz), dz_labels, fontsize=6)
axes[1].set_title(f'|Off-diagonal Crosstalk| (log scale)')
fig.colorbar(im2, ax=axes[1], shrink=0.8)

fig.tight_layout()
pdf_save(fig)
plt.show()

# Print max off-diagonal crosstalk per DZ term
print(f'\nMax off-diagonal crosstalk per input DZ term:')
for i, (j, kfp) in enumerate(dz_term_list):
    row = offdiag[i, :]
    max_xt = np.max(np.abs(row))
    if max_xt > 1e-6:
        worst_idx = np.argmax(np.abs(row))
        j_w, kfp_w = dz_term_list[worst_idx]
        print(f'  z{j}×{fp_zernike_names[kfp]:>7s}: max |crosstalk| = {max_xt:.4e}  '
              f'(worst: z{j_w}×{fp_zernike_names[kfp_w]})')
    else:
        print(f'  z{j}×{fp_zernike_names[kfp]:>7s}: CLEAN (max |crosstalk| < 1e-6)')

### 10.4 V-Mode Recipes for Individual Double-Zernike Terms

For selected double-Zernike terms, show the v-mode amplitude bar charts and verify
that the reconstruction matches only the target term.

In [ ]:
pdf_section('10.4 V-Mode Recipes for DZ Terms')
# Detailed plots for k_fp=1 (constant) DZ terms: z4-z11
r = svd_results[default_dof_set]
n_trunc = r['n_trunc']

dz_kfp1 = [(j, 1) for j in range(4, 16) if (j, 1) in dz_vmode_amps]
n_plots = len(dz_kfp1)

fig, axes = plt.subplots(3, 4, figsize=(18, 11))
for idx, (ax, (j, kfp)) in enumerate(zip(axes.flat, dz_kfp1)):
    a = dz_vmode_amps[(j, kfp)]
    bar_colors = ['steelblue' if abs(v) > 0.01 * np.max(np.abs(a)) else 'lightgray' for v in a]
    ax.bar(np.arange(1, n_trunc + 1), a, color=bar_colors)
    ax.set_xlabel('V-mode m', fontsize=8)
    ax.set_ylabel('Amplitude a_m', fontsize=8)
    achiev = dz_achievable[(j, kfp)]
    ax.set_title(f'z{j}×const  (achiev={achiev:.3f})', fontsize=9)
    ax.grid(alpha=0.2, axis='y')
    ax.axhline(0, color='k', lw=0.5)

fig.suptitle(f'V-mode amplitudes for k_fp=1 (constant) DZ terms ({default_dof_set})', fontsize=12)
fig.tight_layout()
pdf_save(fig)
plt.show()

# Detailed plots for k_fp=2 and k_fp=3 (x-tilt and y-tilt): z4-z8
dz_kfp23 = [(j, 2) for j in range(4, 9)] + [(j, 3) for j in range(4, 9)]
n_plots = len(dz_kfp23)

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for idx, (ax, (j, kfp)) in enumerate(zip(axes.flat, dz_kfp23)):
    a = dz_vmode_amps[(j, kfp)]
    bar_colors = ['coral' if abs(v) > 0.01 * np.max(np.abs(a)) else 'lightgray' for v in a]
    ax.bar(np.arange(1, n_trunc + 1), a, color=bar_colors)
    ax.set_xlabel('V-mode m', fontsize=8)
    ax.set_ylabel('Amplitude a_m', fontsize=8)
    achiev = dz_achievable[(j, kfp)]
    ax.set_title(f'z{j}×{fp_zernike_names[kfp]}  (achiev={achiev:.3f})', fontsize=9)
    ax.grid(alpha=0.2, axis='y')
    ax.axhline(0, color='k', lw=0.5)

fig.suptitle(f'V-mode amplitudes for k_fp=2,3 (tilt) DZ terms ({default_dof_set})', fontsize=12)
fig.tight_layout()
pdf_save(fig)
plt.show()

### 10.5 Verification: Reconstruct Each DZ Term and Check Purity

For each double-Zernike term, reconstruct the wavefront from the v-mode recipe and decompose 
it back into all double-Zernike coefficients. Verify that only the intended term is non-zero.

In [ ]:
pdf_tee_start()

pdf_section('10.5 DZ Term Purity Verification')
# Detailed verification for each DZ term
r = svd_results[default_dof_set]
U_wfs_purity = r['U_wfs']
n_trunc = r['n_trunc']

print(f'DZ Term Purity Check ({default_dof_set}, {n_trunc} retained modes)')
print(f'{"DZ term":>16s} {"self coeff":>11s} {"achievable":>11s} {"max xtalk":>11s} {"xtalk term":>16s} {"pure?":>6s}')
print('-' * 80)

n_clean = 0
n_total = 0
for j, kfp in dz_term_list:
    dz_vec = dz_vectors[(j, kfp)]
    
    # Project onto retained v-modes via lstsq and reconstruct
    a, _, _, _ = np.linalg.lstsq(U_wfs_purity[:, :n_trunc], dz_vec, rcond=None)
    z_recon = U_wfs_purity[:, :n_trunc] @ a
    
    # Decompose reconstruction into all DZ terms
    recon_coeffs = decompose_into_dz(z_recon, dz_vectors)
    
    self_coeff = recon_coeffs.get((j, kfp), 0.0)
    achiev = dz_achievable[(j, kfp)]
    
    # Find worst crosstalk
    max_xtalk = 0.0
    worst_term = None
    for (jj, kkfp), c in recon_coeffs.items():
        if (jj, kkfp) != (j, kfp) and abs(c) > max_xtalk:
            max_xtalk = abs(c)
            worst_term = (jj, kkfp)
    
    is_pure = max_xtalk < 1e-4 * abs(self_coeff) if abs(self_coeff) > 0 else max_xtalk < 1e-10
    n_total += 1
    if is_pure:
        n_clean += 1
    
    worst_str = f'z{worst_term[0]}\u00d7{fp_zernike_names[worst_term[1]]}' if worst_term else 'n/a'
    pure_str = 'YES' if is_pure else 'NO'
    
    print(f'z{j}\u00d7{fp_zernike_names[kfp]:>7s}  {self_coeff:>11.6f}  {achiev:>11.4f}  '
          f'{max_xtalk:>11.2e}  {worst_str:>16s}  {pure_str:>6s}')

print(f'\nSummary: {n_clean}/{n_total} DZ terms have pure (no crosstalk) v-mode recipes')
print(f'Note: "pure" means max |crosstalk| < 1e-4 \u00d7 |self coefficient|')

pdf_tee_stop()


<a id='Normalization-Deep-Dive'></a>
## 11. Normalization Scheme Deep-Dive & Unit Invariance

A focused 2-DOF / 1-Zernike example (Cam_dz vs Cam_rx, pupil Z4) that makes the
normalization algebra transparent.  All schemes set $\tilde{A}=A\,\mathrm{diag}(n_i)$;
only some are **unit-invariant** under $x_i\to\alpha x_i$.  Schemes use
`ofc_svd.normalization_weights_by_scheme`.

| Scheme | $n_i$ | unit-invariant? |
|---|---|---|
| `rf` | $r_i f_i$ | no ($\tilde A\propto\alpha$) |
| `r_only` | $r_i$ | yes |
| `inv_f` | $1/f_i$ | yes |
| `geom_mean` | $\sqrt{r_i/f_i}$ | yes (recommended) |
| `tunable` | $r_i^a f_i^{a-1}$ | yes (any $a$) |

In [ ]:
pdf_section('11. Normalization Scheme Deep-Dive')
pdf_tee_start()

# Minimal example: 2 DOFs (Cam_dz, Cam_rx), 1 pupil Zernike (Z4), 4 WFS.
ns_dof_names = ['Cam_dz', 'Cam_rx']
ns_dof_idx = [labels_50dof.index(n) for n in ns_dof_names]
ns_zn = np.array([4])

ns_ofc = OFCData('lsst', config_dir=ofc_config_dir)
ns_ofc.zn_selected = ns_zn
ns_fa = [ns_ofc.sample_points[s] for s in sensor_name_list]
ns_dz_sens = SensitivityMatrix(ns_ofc)
ns_sens = ns_dz_sens.evaluate(ns_fa, 0.0)[:, ns_zn - 4, :]
ns_A = ns_sens.reshape((n_wfs * len(ns_zn), -1))[:, ns_dof_idx]   # (4, 2)

# r_i, f_i from ofc_svd; n_i per scheme from ofc_svd.
ns_rw, ns_fw = osv.compute_normalization_components(ns_ofc, ns_dz_sens, ns_fa)
r_i = ns_rw[ns_dof_idx]
f_i = ns_fw[ns_dof_idx]
n_stored = ns_ofc.normalization_weights[ns_dof_idx]

print(f'Raw A (4 WFS x {ns_dof_names}):')
print(ns_A)
print(f'\nr_i = {r_i},  f_i = {f_i},  stored n_i = {n_stored}')

ns_schemes = {
    'raw':       np.ones(2),
    'default':   osv.normalization_weights_by_scheme('default', stored=n_stored),
    'rf':        osv.normalization_weights_by_scheme('rf', r_i, f_i),
    'inv_f':     osv.normalization_weights_by_scheme('inv_f', r_i, f_i),
    'geom_mean': osv.normalization_weights_by_scheme('geom_mean', r_i, f_i),
}
print('\nColumn-norm ratio (Cam_dz / Cam_rx) by scheme:')
for name, n in ns_schemes.items():
    A_s = ns_A @ np.diag(n)
    cn = np.linalg.norm(A_s, axis=0)
    print(f'  {name:>10s}: ratio = {cn[0] / cn[1]:.4f}')

pdf_tee_stop()

In [ ]:
# V-mode composition under each normalization scheme.
pdf_section('11.1 V-Modes Under Different Normalizations')
ns_plot = [('raw A', ns_schemes['raw']),
           ('rf (NOT inv.)', ns_schemes['rf']),
           ('inv_f', ns_schemes['inv_f']),
           ('geom_mean', ns_schemes['geom_mean'])]
fig, axes = plt.subplots(1, len(ns_plot), figsize=(5 * len(ns_plot), 4))
for ax, (title, n) in zip(axes, ns_plot):
    A_s = ns_A @ np.diag(n)
    U_s, s_s, Vh_s = np.linalg.svd(A_s, full_matrices=False)
    V_s = Vh_s.T
    im = ax.imshow(V_s, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(s_s)), [f'm={m+1}\nσ={s_s[m]:.3f}' for m in range(len(s_s))],
                  fontsize=8)
    ax.set_yticks(range(len(ns_dof_names)), ns_dof_names)
    ax.set_xlabel('V-mode'); ax.set_title(title, fontsize=10)
    for i in range(V_s.shape[0]):
        for m in range(V_s.shape[1]):
            ax.text(m, i, f'{V_s[i, m]:+.3f}', ha='center', va='center',
                    fontsize=9, color='white' if abs(V_s[i, m]) > 0.5 else 'black')
fig.suptitle('V-Mode Composition Under Different Normalizations', fontsize=13)
fig.tight_layout(); pdf_save(fig); plt.show()

In [ ]:
# Numerical unit-invariance demonstration: re-express Cam_dz in mm (alpha=1000).
pdf_tee_start()
pdf_section('11.2 Unit-Invariance Demonstration')
alpha = 1000.0
A_alt = ns_A.copy(); A_alt[:, 0] *= alpha
r_alt = r_i.copy(); r_alt[0] /= alpha
f_alt = f_i.copy(); f_alt[0] *= alpha

def _maxdiff(M1, M2):
    return np.max(np.abs(M1 - M2))

for scheme in ['rf', 'inv_f', 'geom_mean']:
    A_um = ns_A @ np.diag(osv.normalization_weights_by_scheme(scheme, r_i, f_i))
    A_mm = A_alt @ np.diag(osv.normalization_weights_by_scheme(scheme, r_alt, f_alt))
    d = _maxdiff(A_um, A_mm)
    print(f'  {scheme:>10s}: max|Ã(mm) - Ã(µm)| = {d:.3e}  '
          f'-> {"INVARIANT" if d < 1e-10 else "NOT invariant"}')

print('''
CONCLUSION: the unit-invariant family n_i = r_i^a * f_i^(a-1), a in [0,1],
keeps Ã independent of the physical units chosen for each DOF.  rf (a -> r*f)
fails this.  Recommended: geom_mean, n_i = sqrt(r_i/f_i) (a=1/2) — symmetric
weighting of optical cost (f_i) and mechanical range (r_i).''')
pdf_tee_stop()

<a id='DZ-Field-Patterns'></a>
## 12. Double-Zernike Field Patterns & Physical Impact

Which DOFs produce wavefront patterns described by specific double-Zernike terms
$(j,k)$ — pupil Zernike $Z_j$ whose amplitude varies as focal-plane Zernike $Z_k$
across the field.  The sensitivity matrix is evaluated at the 4 WFS corners
**plus extra field points** to break the rank degeneracy of the 4-corner basis,
then each DOF column is projected onto the focal-plane Zernike basis (built with
`ofc_svd.focal_zernike_at_points` / `make_dz_basis_vector`).

Primary targets: $(5,5)$ and $(6,6)$ — astigmatism in the pupil varying as
astigmatism across the field.

In [ ]:
pdf_section('12. Double-Zernike Field Patterns & Physical Impact')

labels_50dof_full = osv.LABELS_50DOF       # == labels_50dof
dof_units_50 = osv.DOF_UNITS_50            # μm / arcsec per DOF

# Pupil Zernike range and focal-plane order for the DZ decomposition.
dz_jmin, dz_jmax = 4, 11
dz_kmax = 6
dz_targets = [(5, 5), (6, 6), (5, 6), (6, 5), (4, 4)]

# Extra field points (deg) supplementing the 4 WFS corners.  Diagonal points
# (45/135/225/315) are needed to resolve k=5 (astig-45).
extra_field_points = [
    (0.0, 0.0), (0.5, 0.0), (0.0, 0.5), (-0.5, 0.0), (0.0, -0.5),
    (0.35, 0.35), (-0.35, 0.35), (-0.35, -0.35), (0.35, -0.35),
]
extra_field_labels = ['ExtraCtr', 'Extra0', 'Extra90', 'Extra180', 'Extra270',
                      'Extra45', 'Extra135', 'Extra225', 'Extra315']

all_field_angles = [list(a) for a in field_angles] + [list(p) for p in extra_field_points]
fp_labels = [f'WFS{i} ({sensor_name_list[i]})' for i in range(n_wfs)] + extra_field_labels
n_extra = len(extra_field_points)
n_fp = n_wfs + n_extra

# Sensitivity at all field positions: A_3d[s, p, d].
A_3d = dz_sensitivity_matrix.evaluate(all_field_angles, 0.0)[:, zn_idx, :]
n_dof_full = A_3d.shape[2]
print(f'A_3d shape {A_3d.shape}  (n_fp={n_fp}, n_zernike={n_zernike}, n_dof={n_dof_full})')

# Normalized field coordinates and focal-plane Zernike values (ofc_svd).
fp_xy = np.array(all_field_angles)
R_fp = np.max(np.hypot(fp_xy[:, 0], fp_xy[:, 1]))
rho = np.hypot(fp_xy[:, 0], fp_xy[:, 1]) / R_fp
theta = np.arctan2(fp_xy[:, 1], fp_xy[:, 0])
fp_zernike_vals = {k: osv.focal_zernike_at_points(k, rho, theta)
                   for k in range(1, dz_kmax + 1)}

# Rank check of the focal-plane Zernike design matrix.
FP_design = np.column_stack([fp_zernike_vals[k] for k in range(1, dz_kmax + 1)])
_s_fp = np.linalg.svd(FP_design, compute_uv=False)
rank_fp = int(np.sum(_s_fp > 1e-10))
print(f'Focal-plane Zernike basis Z1-Z{dz_kmax}: rank {rank_fp} at {n_fp} positions '
      f'({"resolvable" if rank_fp >= dz_kmax else "DEGENERATE"})')

In [ ]:
# Focal-plane Zernike polynomial values at each field position.
pdf_section('12.1 Focal-Plane Zernike Patterns')
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for k, ax in zip(range(1, dz_kmax + 1), axes.flat):
    vals = fp_zernike_vals[k]
    vmax = max(np.max(np.abs(vals)), 0.01)
    sc = ax.scatter(fp_xy[:, 0], fp_xy[:, 1], c=vals, cmap='seismic',
                    vmin=-vmax, vmax=vmax, s=200, edgecolors='k', zorder=5)
    ax.scatter(fp_xy[:n_wfs, 0], fp_xy[:n_wfs, 1], facecolors='none',
               edgecolors='green', s=350, lw=2, label='WFS', zorder=6)
    ax.scatter(fp_xy[n_wfs:, 0], fp_xy[n_wfs:, 1], facecolors='none',
               edgecolors='orange', s=350, lw=2, marker='s', label='Extra', zorder=6)
    ax.set_title(f'field Z{k} ({osv.FP_ZERNIKE_NAMES[k]})')
    ax.set_xlabel('x (deg)'); ax.set_ylabel('y (deg)')
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    fig.colorbar(sc, ax=ax, shrink=0.8)
axes.flat[0].legend(fontsize=7, loc='lower left')
fig.suptitle(f'Focal-plane Zernike values at {n_fp} field positions', fontsize=12)
fig.tight_layout(); pdf_save(fig); plt.show()

In [ ]:
# Project each DOF's wavefront signature onto the focal-plane Zernike basis.
pdf_tee_start()
pdf_section('12.2 DZ Decomposition & DOF Ranking')
n_j = dz_jmax - dz_jmin + 1
dz_coeff_all = np.zeros((n_dof_full, n_j, dz_kmax))
dz_residual_all = np.zeros((n_dof_full, n_j))
for d in range(n_dof_full):
    for jl, jn in enumerate(range(dz_jmin, dz_jmax + 1)):
        match = np.where(zn == jn)[0]
        if len(match) == 0:
            continue
        a_field = A_3d[:, match[0], d]
        c, *_ = np.linalg.lstsq(FP_design, a_field, rcond=None)
        dz_coeff_all[d, jl, :] = c
        dz_residual_all[d, jl] = np.linalg.norm(a_field - FP_design @ c)

for jt, kt in dz_targets:
    coeffs = dz_coeff_all[:, jt - dz_jmin, kt - 1]
    order = np.argsort(np.abs(coeffs))[::-1]
    print(f'\n--- (j={jt}, k={kt}): pupil Z{jt} × field Z{kt} '
          f'({osv.FP_ZERNIKE_NAMES[kt]}) ---')
    print(f'{"rank":>4s} {"DOF":>12s} {"coeff":>12s} {"resid":>10s}')
    for ri in range(min(8, n_dof_full)):
        d = order[ri]
        if abs(coeffs[d]) < 1e-10:
            break
        print(f'{ri+1:>4d} {labels_50dof[d]:>12s} {coeffs[d]:>12.6f} '
              f'{dz_residual_all[d, jt - dz_jmin]:>10.2e}')
pdf_tee_stop()

In [ ]:
# |DZ coefficient| heatmap (significant DOFs) and top-DOF bar charts.
pdf_section('12.3 DZ Coefficient Heatmap & Top DOFs')
dz_flat = dz_coeff_all.reshape(n_dof_full, n_j * dz_kmax)
dz_flat_labels = [f'z{jn},k{k}' for jn in range(dz_jmin, dz_jmax + 1)
                  for k in range(1, dz_kmax + 1)]
target_cols = [(jt - dz_jmin) * dz_kmax + (kt - 1) for jt, kt in dz_targets]
max_target = np.max(np.abs(dz_flat[:, target_cols]), axis=1)
sig = np.where(max_target > 0.01 * np.max(max_target))[0]

fig, ax = plt.subplots(figsize=(18, max(6, len(sig) * 0.3)))
data = np.abs(dz_flat[sig, :])
vmax = max(np.max(data), 1e-12)
im = ax.imshow(data, aspect='auto', cmap='YlOrRd',
               norm=colors.LogNorm(vmin=max(1e-8, vmax * 1e-4), vmax=vmax))
ax.set_yticks(range(len(sig)), [f'{labels_50dof[d]} (#{d})' for d in sig], fontsize=7)
ax.set_xticks(range(len(dz_flat_labels)), dz_flat_labels, rotation=90, fontsize=6)
ax.set_xlabel('Double-Zernike term (j, k)'); ax.set_ylabel('DOF')
ax.set_title(f'|Double-Zernike coefficient| per DOF (log, {n_fp} field positions)')
for tc in target_cols:
    ax.axvline(tc, color='cyan', lw=1.5, alpha=0.7, ls='--')
fig.colorbar(im, ax=ax, shrink=0.8, label='|coefficient|')
fig.tight_layout(); pdf_save(fig); plt.show()

fig, axes = plt.subplots(1, len(dz_targets), figsize=(5 * len(dz_targets), 6))
if len(dz_targets) == 1:
    axes = [axes]
n_show = 12
for (jt, kt), ax in zip(dz_targets, axes):
    coeffs = dz_coeff_all[:, jt - dz_jmin, kt - 1]
    order = np.argsort(np.abs(coeffs))[::-1][:n_show]
    vals = coeffs[order]
    ax.barh(range(n_show), vals,
            color=['steelblue' if v >= 0 else 'coral' for v in vals])
    ax.set_yticks(range(n_show), [labels_50dof[d] for d in order], fontsize=8)
    ax.set_xlabel('DZ coeff')
    ax.set_title(f'(j={jt}, k={kt})\nZ{jt}×{osv.FP_ZERNIKE_NAMES[kt]}', fontsize=10)
    ax.axvline(0, color='k', lw=0.5); ax.invert_yaxis(); ax.grid(alpha=0.2, axis='x')
fig.suptitle(f'Top DOFs for target double-Zernike terms ({n_fp} field positions)',
             fontsize=13, y=1.02)
fig.tight_layout(); pdf_save(fig); plt.show()

In [ ]:
# Expected field pattern (top) vs actual sensitivity of top DOFs (bottom).
pdf_section('12.4 Astigmatic Field-Pattern Verification (Z5, Z6)')
for jt, kt in [(5, 5), (6, 6)]:
    coeffs = dz_coeff_all[:, jt - dz_jmin, kt - 1]
    order = np.argsort(np.abs(coeffs))[::-1]
    p_j = np.where(zn == jt)[0][0]
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    for ci, k in enumerate([4, 5, 6]):
        ax = axes[0, ci]
        vals = fp_zernike_vals[k]
        vmax = max(np.max(np.abs(vals)), 0.01)
        sc = ax.scatter(fp_xy[:, 0], fp_xy[:, 1], c=vals, cmap='seismic',
                        vmin=-vmax, vmax=vmax, s=150, edgecolors='k')
        ax.set_title(f'Expected field Z{k} ({osv.FP_ZERNIKE_NAMES[k]})', fontsize=9)
        ax.set_aspect('equal'); ax.grid(alpha=0.3); fig.colorbar(sc, ax=ax, shrink=0.8)
    for ri in range(3):
        d = order[ri]
        ax = axes[1, ri]
        a_field = A_3d[:, p_j, d]
        vmax = max(np.max(np.abs(a_field)), 1e-10)
        sc = ax.scatter(fp_xy[:, 0], fp_xy[:, 1], c=a_field, cmap='seismic',
                        vmin=-vmax, vmax=vmax, s=150, edgecolors='k')
        ax.set_title(f'Z{jt}: {labels_50dof[d]}  DZ({jt},{kt})={coeffs[d]:.4f}', fontsize=9)
        ax.set_aspect('equal'); ax.grid(alpha=0.3)
        fig.colorbar(sc, ax=ax, shrink=0.8,
                     label=f'µm/{dof_units_50[d]}')
    fig.suptitle(f'Pupil Z{jt}: expected field pattern (top) vs actual sensitivity (bottom)',
                 fontsize=12)
    fig.tight_layout(); pdf_save(fig); plt.show()

In [ ]:
# Physical impact: Z5/Z6 wavefront change at realistic DOF perturbations.
pdf_tee_start()
pdf_section('12.5 Physical Impact at Field Points')
# Perturbation: 1 arcsec rotations, 0.5 µm bending modes, 1.0 µm translations.
dof_perturbation = np.array([
    1.0 if dof_units_50[d] == 'arcsec' else (0.5 if d >= 10 else 1.0)
    for d in range(n_dof_full)])
p_z5 = np.where(zn == 5)[0][0]
p_z6 = np.where(zn == 6)[0][0]
print('Bending 0.5 µm | translation 1.0 µm | rotation 1.0 arcsec\n')
for jt, kt in [(5, 5), (6, 6), (5, 6), (6, 5)]:
    coeffs = dz_coeff_all[:, jt - dz_jmin, kt - 1]
    order = np.argsort(np.abs(coeffs))[::-1][:5]
    print(f'--- DZ({jt},{kt}) ---')
    print(f'{"DOF":>12s} {"unit":>7s} {"amp":>6s} {"max|ΔZ5|":>10s} {"max|ΔZ6|":>10s}')
    for d in order:
        if abs(coeffs[d]) < 1e-10:
            continue
        amp = dof_perturbation[d]
        dz5 = A_3d[:, p_z5, d] * amp
        dz6 = A_3d[:, p_z6, d] * amp
        print(f'{labels_50dof[d]:>12s} {dof_units_50[d]:>7s} {amp:>6.2f} '
              f'{np.max(np.abs(dz5)):>10.4f} {np.max(np.abs(dz6)):>10.4f}')
    print()
pdf_tee_stop()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, (p_z, zl) in zip(axes, [(p_z5, 'Z5'), (p_z6, 'Z6')]):
    max_dz = np.array([np.max(np.abs(A_3d[:, p_z, d])) * dof_perturbation[d]
                       for d in range(n_dof_full)])
    order = np.argsort(max_dz)[::-1][:15]
    bar_colors = ['steelblue' if d < 5 else 'coral' if d < 10
                  else 'mediumseagreen' if d < 30 else 'orchid' for d in order]
    ax.barh(range(len(order)), max_dz[order], color=bar_colors)
    ax.set_yticks(range(len(order)),
                  [f'{labels_50dof[d]} ({dof_perturbation[d]:.1f}{dof_units_50[d]})'
                   for d in order], fontsize=8)
    ax.set_xlabel(f'max |Δ{zl}| across field (µm)')
    ax.set_title(f'Top DOFs by max |Δ{zl}|', fontsize=11)
    ax.invert_yaxis(); ax.grid(alpha=0.2, axis='x')
fig.suptitle('Physical impact: which DOFs produce the most Z5/Z6 change?', fontsize=13)
fig.tight_layout(); pdf_save(fig); plt.show()

In [ ]:
# Summary table of best DOFs per target DZ term.
pdf_tee_start()
pdf_section('12.6 Summary')
print(f'DOFs producing astigmatic field patterns ({n_fp} field positions, '
      f'focal basis rank {rank_fp})')
print(f'{"(j,k)":>8s} {"best DOF":>12s} {"coeff":>12s} {"resid":>10s} {"2nd":>12s} {"coeff":>12s}')
for jt, kt in dz_targets:
    coeffs = dz_coeff_all[:, jt - dz_jmin, kt - 1]
    order = np.argsort(np.abs(coeffs))[::-1]
    d1, d2 = order[0], order[1]
    print(f'({jt:2d},{kt:2d}) {labels_50dof[d1]:>12s} {coeffs[d1]:>12.6f} '
          f'{dz_residual_all[d1, jt - dz_jmin]:>10.2e} {labels_50dof[d2]:>12s} '
          f'{coeffs[d2]:>12.6f}')
pdf_tee_stop()

In [ ]:
# Close the PDF file
pdf.close()
print(f'PDF saved: {pdf_filename}')